## Step 2 · Processing & Extraction

Reads the transcript table from `step1_read.ipynb`, extracts all study variables, derives analytic variables, applies QC rules, and outputs analysis-ready files.

| | |
|---|---|
| **Reads** | `output/raw_transcripts.csv` |
| **Outputs** | `output/metadata.csv`, `metadata_qc_flags.csv`, `metadata_key_variables.csv` |

**Derived variables**
- `Age_at_Arrival` = Year_of_Arrival − Birth_Year
- `Generational_Status` = 1st / 1.5 / 1.75 / 2nd gen
- `Arrival_Wave` = Wave 1 / 2 / 3  ·  Inclusion filter: Year_of_Arrival ≤ 1995

---

## Block 0 · Setup — Install Dependencies & Imports

**Run this first.**

In [37]:
# ── STEP 2: CLEAN PATH SETUP ──────────────────────────────────────────────
from pathlib import Path
import pandas as pd
import numpy as np
import re


# from google.colab import drive
# drive.mount("/content/drive")

# Prefer local runtime copy from Step 1; fall back to Google Drive copy
INPUT_FILE = Path("interview_data.xlsx")
OUTPUT_FILE = Path("metadata.xlsx")
# DRIVE_INPUT = Path("/content/drive/MyDrive/803_oralhistory/interview_data.xlsx")
# LOCAL_INPUT = Path("/output/raw_transcripts.csv")
# DRIVE_INPUT = Path("/content/drive/MyDrive/803_oralhistory/output/raw_transcripts.csv")

# if LOCAL_INPUT.exists():
#     INPUT_FILE = LOCAL_INPUT
# # elif DRIVE_INPUT.exists():
# #     INPUT_FILE = DRIVE_INPUT
# else:
#     raise FileNotFoundError(
#         "Step 1 output not found in either:\n"
#         f" - {LOCAL_INPUT}\n"
#         # f" - {DRIVE_INPUT}"
#     )

# # OUTPUT_DIR = Path("/content/drive/MyDrive/803_oralhistory/output")
# # OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# OUTPUT_FILE        = OUTPUT_DIR / "metadata.csv"
# FLAGS_FILE         = OUTPUT_DIR / "metadata_qc_flags.csv"
# ALL_VARIABLES_FILE = OUTPUT_DIR / "metadata_all_variables.csv"
# KEY_VARIABLES_FILE = OUTPUT_DIR / "metadata_key_variables.csv"
# TRACE_FILE         = OUTPUT_DIR / "metadata_missing_trace.csv"
# VERDICT_FILE       = OUTPUT_DIR / "metadata_missing_verdict.csv"

# print(f"✓ Using input: {INPUT_FILE}")
# print(f"✓ Saving outputs to: {OUTPUT_DIR}")


In [38]:
# ── STEP 2: LOAD STEP 1 HELPER ────────────────────────────────────────────
def load_step1_output(input_file):
    input_file = Path(input_file)
    print("DEBUG load_step1_output path:", input_file)
    if not input_file.exists():
        raise FileNotFoundError(
            f"Step 1 output not found: {input_file}. Run the Step 1 notebook first."
        )
    return pd.read_excel(input_file)


## Block 1 · Config & Path Setup

Path constants and helpers. Auto-detects project root from the notebook's location.

## Block 2 · Load Step 1 Output

Load `raw_transcripts.csv` saved by step1 directly into `df`.

## Block 3 · Supplementary Extraction — Core Patterns & Helpers

Regex patterns and helper functions that fill in fields step 1 missed (name, gender, birth year, arrival year, etc.).

In [39]:
# 1b. Supplementary extraction — fills gaps step1 missed
#     Runs only on rows that still have missing key fields.
#     Uses only the CSV produced by step1_read.py.
# ---------------------------------------------------------------------------

_SUPP_ARRIVAL_PATTERNS = [
    r"\b((?:19|20)\d{2})\s+Left\s+Vietnam\b[^.?\n]{0,140}\bArrived\s+at\s+(?:Philippines|Guam|Camp\s+Pendleton|Pennsylvania)\b",
    r"\b((?:19|20)\d{2})\s+(?:Mr\.?|Mrs\.?|Ms\.?|Miss)?\s*[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,40}\s+"
    r"(?:and\s+(?:his|her)\s+family\s+are\s+sponsored\s+to\s+(?:go|come)\s+to\s+America\b[^.?\n]{0,80}\barrives?\s+in\s+[A-Z]"
    r"|(?:left|leaves)\s+(?:Saigon|Vietnam)\b[^.?\n]{0,120}\barrives?\s+in\s+(?:Pennsylvania|America|the\s+United\s+States))",
    r"\b((?:19|20)\d{2})\s+(?:Mr\.?|Mrs\.?|Ms\.?|Miss)?\s*[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,40}\s+"
    r"(?:came|arrived|moved|resettled|immigrated)\s+(?:to|in)\s+(?:the\s+)?(?:US|U\.S\.|United\s+States|America)\b",
    r"\b((?:19|20)\d{2})\s+(?:Mr\.?|Mrs\.?|Ms\.?|Miss)?\s*[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,40}\s+"
    r"(?:moved|came|arrived|resettled)\s+to\s+[A-Z][A-Za-z .'-]{1,40},\s*California\b",
    r"\b((?:19|20)\d{2})\s+(?:moved|came|arrived|resettled)\s+to\s+[A-Z][A-Za-z .'-]{1,40},\s*California\b",
    r"\b((?:19|20)\d{2})\s+(?:Mr\.?|Mrs\.?|Ms\.?|Miss)?\s*[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,40}\s+"
    r"came\s+to\s+(?:the\s+)?(?:US|U\.S\.|United\s+States|America)\s+by\s+boat\b",
    r"\b((?:19|20)\d{2})\s+Sponsored\b[^.?\n]{0,120}\b(?:California|America|United\s+States|U\.S\.|US)\b",
    r"\b((?:19|20)\d{2})\s+Sponsored\s+with\b[^.?\n]{0,120}\b(?:California|America|United\s+States|U\.S\.|US)\b",
    # Direct U.S.-arrival statements
    r"\b(?:my\s+parents?|my\s+family)\s+(?:immigrated|came\s+over|came|arrived|moved|resettled)\s+"
    r"(?:to|in)\s+(?:the\s+)?(?:US|U\.S\.|United\s+States|America|States)\s+(?:in\s+)?((?:19|20)\d{2})\b",
    r"\bcame\s+to\s+(?:the\s+)?(?:US|U\.S\.|America|United\s+States)\b[^.?\n]{0,40}?\b((?:19|20)\d{2})\b",
    r"\barrived\s+(?:here|to\s+(?:the\s+)?(?:US|U\.S\.|America|United\s+States)|in\s+(?:the\s+)?(?:US|U\.S\.|America|United\s+States))\b[^.?\n]{0,60}?\b((?:19|20)\d{2})\b",
    r"\b(?:came|moved|resettled)\s+here\b[^.?\n]{0,40}?\b((?:19|20)\d{2})\b",
    r"\b(?:the\s+beginning\s+of|beginning\s+of)\s+((?:19|20)\d{2})\b[^.?\n]{0,40}\bI\s+came\s+over\s+here\b",
    r"\b(?:the\s+)?beginning\s+((?:19|20)\d{2})\b[^.?\n]{0,40}\bI\s+came\s+over\s+here\b",
    r"\bI\s+came\s+over\s+here\b[^.?\n]{0,40}?\b((?:19|20)\d{2})\b",
    r"\bI\s+came\b[^.?\n]{0,40}?\b(19[6-9][0-9]|20[0-2][0-9])\b",
    r"\b(?:through|under)\s+(?:the\s+)?(?:ODP|Orderly\s+Departure\s+Program)\b[^.?\n]{0,60}\b((?:19|20)\d{2})\b",
    r"\b((?:19|20)\d{2})\b[^.?\n]{0,50}\b(?:through|under)\s+(?:the\s+)?(?:ODP|Orderly\s+Departure\s+Program)\b",
    r"\b(?:came\s+to\s+America|came\s+to\s+the\s+U\.?S\.?|came\s+to\s+the\s+United\s+States|arrived\s+in\s+America|arrived\s+in\s+the\s+United\s+States)\s+on\s+(?:[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+)?((?:19|20)\d{2})\b",
    r"\bI\s+(?:believe|think)\s+it\s+was\s+(?:around|about)\s+((?:19|20)\d{2})\b",
    r"\bit\s+was\s+about\s+((?:19|20)\d{2})\b",
    r"\blanded\s+(?:in\s+)?(?:the\s+)?(?:US|U\.S\.|United\s+States|America|States|[A-Z][a-z]{2,15})\s+(?:in\s+)?((?:19|20)\d{2})\b",
    r"\bentered\s+(?:the\s+)?(?:US|U\.S\.|United\s+States|America|country)\s+(?:in\s+)?((?:19|20)\d{2})\b",
    r"\b(?:my\s+(?:family|parents?|mother|father|husband|wife)|we)\s+(?:came|arrived|moved|resettled)\s+"
    r"(?:here|to\s+(?:the\s+)?(?:US|U\.S\.|America|States))\s+(?:in\s+)?((?:19|20)\d{2})\b",
    r"\b(?:we\s+|I\s+)?(?:left|got\s+out\s+of|were\s+released\s+from|exited)\s+(?:the\s+)?(?:refugee\s+)?camp\s+"
    r"(?:in\s+)?((?:19|20)\d{2})\b",
    r"\b(?:flew|flown|taken\s+by\s+(?:a\s+)?(?:plane|flight)|took\s+a\s+(?:plane|flight))\s+"
    r"(?:to\s+)?(?:the\s+)?(?:US|U\.S\.|United\s+States|America)\s+(?:in\s+)?((?:19|20)\d{2})\b",
    r"\b(?:was\s+admitted|granted\s+entry|received\s+(?:a\s+)?(?:visa|green\s+card|parole))\s+"
    r"(?:to\s+(?:the\s+)?(?:US|America|States)\s+)?(?:in\s+)?((?:19|20)\d{2})\b",
    r"\b(?:I|we)\s+got\s+to\s+(?:the\s+)?(?:US|U\.S\.|America|States|[A-Z][a-z]{2,15})\s+(?:in\s+)?((?:19|20)\d{2})\b",
    r"\buntil\s+((?:19|20)\d{2})\s+and\s+then\s+(?:we|I)\s+got\s+a\s+visa\s+from\s+(?:the\s+)?(?:US|U\.S\.|United\s+States|America)\b",
    # resettlement / sponsorship phrases
    r"\b(?:resettled|sponsored|placed)\s+(?:by\s+\S+\s+)?(?:in\s+(?:the\s+)?(?:US|America|States)\s+)?(?:in\s+)?((?:19|20)\d{2})\b",
    r"\b(?:I|we)\s+(?:came|moved|arrived)\s+(?:here\s+)?(?:in\s+)?((?:19|20)\d{2})\b",
    r"\b(?:I|he|she|we)\s+(?:first\s+)?came\s+to\s+(?:the\s+)?(?:US|U\.S\.|America|United\s+States)\s+(?:in\s+)?((?:19|20)\d{2})\b",
    r"\b((?:19|20)\d{2})\b[^.?\n]{0,120}\barrived\s+in\s+(?:the\s+)?(?:United\s+States|US|U\.S\.|America)\b",
    # "since [year]" phrasing ("I've been here since 1979")
    r"\b(?:I(?:'ve|\s+have)\s+been\s+(?:here|in\s+(?:the\s+)?(?:US|America|States))\s+since\s+)((?:19|20)\d{2})\b",
    r"\bbefore\s+I\s+got\s+into\s+the\s+United\s+States[^.?\n]{0,80}\b(?:when\s+I\s+was\s+)?(\d{1,2})\b",
    r"\b(?:came|arrived|got\s+here|moved\s+here)\s+(?:when\s+I\s+was\s+(?:only\s+|about\s+|around\s+)?)?(\d{1,2})(?:\s+years?\s+old)?\b",
    r"\bI\s+was\s+(?:only\s+|about\s+|around\s+)?(\d{1,2})(?:\s+years?\s+old)?\s+when\s+(?:we|I)\s+(?:came|arrived|got\s+here|moved\s+here)\b",
]

# Patterns that imply a *fixed* arrival year without a capture group.
# Each tuple is (regex, implied_year).
_SUPP_ARRIVAL_FIXED = [
    # Fall of Saigon / April 30 references → 1975
    (r"\b(?:after\s+(?:the\s+)?fall\s+of\s+Saigon|following\s+(?:the\s+)?fall\s+of\s+Saigon"
     r"|April\s+30(?:th)?,?\s*1975|after\s+(?:April|the)\s+(?:30|thirtieth)"
     r"|after\s+(?:the\s+)?communist\s+(?:take\s*over|takeover|victory)\s+(?:of\s+South\s+Vietnam)?"
     r"|when\s+Saigon\s+fell)\b",
     1975),
    # 1975 evacuation route through U.S. military camps
    (r"\b(?:1975[^.?\n]{0,140}\b(?:Camp\s+Pendleton|Guam|USS\s+Midway|helicopter|chopper|evacuat(?:ed|ion))"
     r"|(?:Camp\s+Pendleton|Guam|USS\s+Midway|helicopter|chopper)[^.?\n]{0,140}\b1975)\b",
     1975),
    (r"\b(?:USS\s+Midway|Camp\s+Pendleton|Guam|San\s+Francisco)[^.?\n]{0,180}\b(?:1975|fall\s+of\s+Saigon|April\s+29|April\s+30)\b",
     1975),
]

_SUPP_BIRTH_YEAR_PATTERNS = [
    # Direct narrator statements, including month+year with or without a space:
    r"\bI\s+was\s+born\s+in\s+(?:[A-Z][a-z]+\s*)?(18\d{2}|19\d{2}|20\d{2})\b",
    r"\bborn\s+in\s+(?:[A-Z][a-z]+\s*)?(18\d{2}|19\d{2}|20\d{2})\b",

    r"\b((?:18|19|20)\d{2})\s+Born\s+in\s+[A-ZÀ-Ỹ]",
    r"\b((?:18|19|20)\d{2})\s+(?:Mr\.?|Mrs\.?|Ms\.?|Miss)?\s*[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,40}\s+(?:is\s+)?born\b",
    r"\bmyself\s+\[((?:18|19|20)\d{2})\]\b",
    r"\b(?:\d{1,2})/(?:\d{1,2})/((?:18|19|20)\d{2})\b",
    r"\bwhere\s+and\s+when\s+were\s+you\s+born[\s\S]{0,250}?\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bdate\s+of\s+birth[,:\s-]+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+(\d{2})\b",
    r"\bcan\s+you\s+tell\s+me\s+your\s+date\s+of\s+birth\?[\s\S]{0,120}?\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bcan\s+you\s+tell\s+me\s+your\s+date\s+of\s+birth\?\s*(?:[A-Z][A-Za-zÀ-Ỹà-ỹ.' -]{1,40}:\s*)?[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bmy\s+date\s+of\s+birth\s+is\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bmy\s+real\s+birthday\s+is\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bcan\s+you\s+tell\s+me\s+your\s+date\s+of\s+birth[^.?\n]{0,80}?\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\b(?:where|when)\s+and\s+(?:where|when)\s+were\s+you\s+born[^.?\n]{0,80}\bI\s+was\s+born[^.?\n]{0,80}\.\s*((?:18|19|20)\d{2})\b",
    r"\bI\s+was\s+born\s+in\s+[A-ZÀ-Ỹa-zà-ỹ\s]{2,50}\.\s*((?:18|19|20)\d{2})\b",
    r"\bwas\s+born\s+exactly\s+today[^.?\n]{0,80}\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\b(?:my\s+)?birthday[^.?\n]{0,120}\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\s+in\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{1,50}\b",
    r"\b(?:in\s+)?((?:18|19|20)\d{2})\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?\b",
    r"\b(?:Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\.?\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bborn\s+(?:on\s+)?(?:Christmas\s+Day,\s+)?[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bborn\s+on\s+the\s+\d{1,2}(?:st|nd|rd|th)\s+of\s+[A-Z][a-z]+(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bborn\s+in\s+the\s+(?:spring|summer|fall|autumn|winter)\s+of\s+((?:18|19|20)\d{2})\b",
    r"\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\s*,\s*in\s+[A-ZÀ-Ỹa-zà-ỹ]",
    r"\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\s*,\s+and\s+I\s+was\s+born\b",
    r"\bI\s+(?:was|am)\s+born\s+(?:on\s+)?[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bI\s+born\s+in\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{1,80}\b[^.?\n]{0,120}\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bI\s+born\s+in[\s\S]{0,220}?\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bI\s+was\s+born\s+at\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{1,40}\s+in\s+((?:18|19|20)\d{2})\b",
    r"\bI\s+was\s+born\s+in\s+((?:18|19|20)\d{2})\s+in\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{1,50}\b",
    r"\bI\s+was\s+born\s+in\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{1,50}\s+in\s+((?:18|19|20)\d{2})\b",
    r"\bI\s+was\s+born\s+in\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{1,50},\s*(?:Vietnam|Việt\s+Nam)\s*,?\s*in\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bI\s+was\s+born\s+in(?:\s+actually)?\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{1,40}\s+(?:Vietnam|Việt\s+Nam)\s+((?:18|19|20)\d{2})\b",
    r"\bI\s+was\s+born\s+in\s+Vietnam\s+and\s+in\s+((?:18|19|20)\d{2})\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?\b",
    r"\bI\s+(?:was|am)\s+born\s+in\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{1,50},\s*(?:Vietnam|Việt\s+Nam)\s*,?\s+in\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bI\s+was\s+born\s+in\s+[A-Z][a-z]+\s+((?:18|19|20)\d{2})\s+in\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{1,50}\b",
    r"\bin\s+((?:18|19|20)\d{2}),\s*\d{1,2}(?:st|nd|rd|th)?\s+of\s+[A-Z][a-z]+\b",
    r"\bI\s+was\s+born\s+on\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19)\d{2})\b",
    r"\bI\s+was\s+born\s+in\s+(?:Vietnam|[A-ZÀ-Ỹa-zà-ỹ\s]+)\s+on\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bI\s+was\s+born\s+in\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]+,\s*(?:Vietnam|Việt\s+Nam),\s*in\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bI\s+am\s+born\s+in\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bI\s+am\s+born\s+in\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{1,50},\s*(?:Vietnam|Việt\s+Nam)\s*,?\s+in\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bmy\s+birthday\s+is\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bmy\s+name\s+is\b[^.?\n]{0,80}\.\s*[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19)\d{2})\b",
    r"\b[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,40},\s*((?:18|19)\d{2}),\s*[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?\b",
    r"\b(?:name\s+is|I(?:'m|\s+am)\s+called)\b[^.?\n]{0,80}\b((?:18|19)\d{2})\b",
    r"\b(?:my\s+day\s+of\s+birth\s+is|date\s+of\s+(?:birth|mrach))\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19)\d{2})\b",
    r"\bdate\s+of\s+birth[:\s-]+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19|20)\d{2})\b",
    r"\bdate\s+of\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19)\d{2})\b",
    r"\b(?:my\s+date\s+of\s+birth\s+is|date\s+of\s+birth\s+is)\s+the\s+(?:first|second|third|fourth|fifth|sixth|seventh|eighth|ninth|tenth|eleventh|twelfth|thirteenth|fourteenth|fifteenth|sixteenth|seventeenth|eighteenth|nineteenth|twentieth|twenty[- ]first|twenty[- ]second|twenty[- ]third|twenty[- ]fourth|twenty[- ]fifth|twenty[- ]sixth|twenty[- ]seventh|twenty[- ]eighth|twenty[- ]ninth|thirtieth|thirty[- ]first)\s+of\s+[A-Z][a-z]+\s+((?:18|19)\d{2})\b",
    r"\bI\s+was\s+born\s+in\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+((?:18|19)\d{2})\b",
    r"\bsinh\s+ngày\s+\d{1,2}\s+tháng\s+\d{1,2}\s+năm\s+((?:18|19)\d{2})\b",
    r"\b(?:I\s+was\s+)?born\s+in\s+((?:18|19)\d{2})\b",
    r"\b(?:I\s+was\s+)?born\s+in\s+[A-ZÀ-Ỹa-zà-ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{1,50}?,\s*((?:18|19)\d{2})\b",
    # NOTE: removed loose "YYYY in [Place]" pattern — matches historical event
    # phrases like "April 30, 1975 in South Vietnam" giving wrong birth years.
    r"\bI\s+was\s+born\s+((?:18|19)\d{2})\s+in\b",
    r"(?:when\s+were\s+you\s+born|what\s+year\s+(?:were\s+you\s+born|is\s+your\s+birth(?:day|date)?))"
    r"[^.?\n]{0,150}?\b((?:18|19)\d{2})\b",
    r"(?:^|\n)\s*[Bb]orn[:\s]+((?:18|19)\d{2})\b",
    r"(?:^|\n)\s*[Bb]irth\s*[Yy]ear[:\s]+((?:18|19)\d{2})\b",
    r"\bI\s+was\s+born\s+((?:18|19)\d{2})\b",
]

_SUPP_BIRTHPLACE_PATTERNS = [
    r"\b(?:18|19|20)\d{2}\s+Born\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?(?:,\s*(?:Vietnam|Việt\s+Nam|South\s+Vietnam|North\s+Việt\s+Nam|North\s+Vietnam))?)\b",
    r"\b(?:18|19|20)\d{2}\s+(?:Mr\.?|Mrs\.?|Ms\.?|Miss)?\s*[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,40}\s+(?:is\s+)?born\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?(?:,\s*(?:Vietnam|Việt\s+Nam|South\s+Vietnam|North\s+Vietnam))?)\b",
    r"\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+(?:18|19|20)\d{2}\s*,\s+and\s+I\s+was\s+born\s+(?:in\s+)?([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?(?:,\s*(?:Vietnam|Việt\s+Nam))?)\b",
    r"\b(?:I\s+was\s+)?born\s+in\s+(?:18|19|20)\d{2},\s*([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35}(?:,\s*(?:Vietnam|Việt\s*Nam|South\s+Vietnam|North\s+Vietnam))?)\b",
    r"\bI\s+was\s+born\s+at\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?(?:,\s*(?:North\s+Việt\s+Nam|North\s+Vietnam|South\s+Vietnam|Vietnam|Việt\s+Nam))?)\s+in\s+(?:18|19|20)\d{2}\b",
    r"\bI\s+was\s+born\s+at\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35}(?:,\s*(?:North\s+Việt\s*Nam|North\s+Vietnam|South\s+Vietnam|Vietnam|Việt\s*Nam))?)\b",
    r"\bI\s+was\s+born\s+in\s+(?:18|19|20)\d{2}\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?(?:\s+in\s+(?:Central|South|North)\s+Vietnam|\s*,\s*(?:Vietnam|Việt\s+Nam)|\s+Province)?)\b",
    r"\b(?:I\s+was\s+)?born\s+in\s+(?:18|19|20)\d{2}\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35}\s+(?:Vietnam|Việt\s*Nam|South\s+Vietnam|North\s+Vietnam))\b",
    r"\bI\s+was\s+born\s+in\s+[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+(?:18|19|20)\d{2},\s*([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35}(?:,\s*(?:Vietnam|Việt\s*Nam|South\s+Vietnam|North\s+Vietnam))?)\b",
    r"\bI\s+was\s+born\s+(?:on\s+)?[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+(?:18|19|20)\d{2}\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35}(?:,\s*(?:Vietnam|Việt\s*Nam|South\s+Vietnam|North\s+Vietnam))?)\b",
    r"\b(?:I\s+was\s+|I\s+am\s+|)\s*born\s+on\s+\d{1,2}(?:st|nd|rd|th)?\s+of\s+[A-Z][a-z]+,\s*(?:18|19|20)\d{2}\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35}(?:,\s*(?:Vietnam|Việt\s*Nam|South\s+Vietnam|North\s+Vietnam))?)\b",
    r"\b(?:I\s+was\s+|I\s+am\s+|)\s*born\s+on\s+the\s+\d{1,2}(?:st|nd|rd|th)?\s+of\s+[A-Z][a-z]+,\s*(?:18|19|20)\d{2}\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35}(?:,\s*(?:Vietnam|Việt\s*Nam|South\s+Vietnam|North\s+Vietnam))?)\b",
    r"\b(?:I\s+was\s+|I\s+am\s+|)\s*born\s+on\s+[A-Z][a-z]+\s+\d{1,2},?\s*(?:18|19|20)\d{2}\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35})(?:\s*,?\s*a\s+city\b|\s*,?\s*it(?:'s|\s+is)\b|[.,])",
    r"\b(?:I\s+was\s+|I\s+am\s+|)\s*born\s+in\s+[A-Z][a-z]+\s+\d{1,2},?\s*(?:18|19|20)\d{2}\s+in\s+Vietnam\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35})(?:,\s*particularly|\b)",
    r"\bI\s+was\s+born\s+(?:on\s+)?(?:Christmas\s+Day,\s+)?[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+(?:18|19|20)\d{2}\.\s+And\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?(?:,\s*(?:Vietnam|Việt\s*Nam|South\s+Vietnam|North\s+Vietnam))?)\b",
    r"\b[A-Z][a-z]+\s+\d{1,2}(?:st|nd|rd|th)?(?:,)?\s+(?:18|19|20)\d{2}\s*,\s*in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?(?:\s+city|\s+province)?)(?:\.|,|$)",
    r"\bI\s+was\s+born\s+and\s+grew\s+up\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?(?:\s+city|\s+province)?)(?:\s+of\s+(?:central|south|north)\s+Vietnam|\b)",
    r"\bI\s+was\s+born\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35})\.\s+It(?:'s|\s+is)\s+in\s+(?:Central|South|North)\s+Vietnam\b",
    r"\bI\s+was\s+born\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?,\s*[A-Z][a-z]+)\b",
    r"\bI\s+myself,\s*was\s+born\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?)(?:\.|,|\s+and\b)",
    r"\bcity\s+I\s+was\s+born\s+in\s+is\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35})\b",
    r"\bplace\s+of\s+birth[,:\s]+\s*([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?,\s*(?:Vietnam|Việt\s+Nam))\b",
    r"\bplace\s+of\s+birth\?\s*(?:[A-Z][A-Za-zÀ-Ỹà-ỹ.' -]{1,24}:\s*)?([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?(?:,\s*(?:Vietnam|Việt\s*Nam|South\s+Vietnam|North\s+Vietnam))?)\b",
    r"\b([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?,\s*(?:Vietnam|Việt\s+Nam))\b[^.?\n]{0,100}\bthat(?:'s|\s+is)\s+where\s+I\s+(?:grew|grown)\s+and\s+born\b",
    r"\bmy\s+birth\s+place\s+is\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?)(?:,|\s+which|\s+in\s+Vietnam|\s+in\s+Việt\s+Nam|\.|$)",
    r"\bborn\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?\s+in\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?\s+province)\b",
    r"\bborn\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,25},\s*[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}\s+Province)\b",
    r"\bborn\s+in\s+\d{4}\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?\s+province)\b",
    r"\bborn\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?)\.\s*(?:Vietnam|Việt\s+Nam)\b",
    r"\bthe\s+north\s+of\s+Vietnam\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35})\b",
    r"\b([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,25},\s*[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,25},\s*(?:Vietnam|Việt\s+Nam))\b",
    r"\bfrom\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}),\s+now\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}\b",
    r"\bI\s+was\s+born\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}),\s+a\s+(?:province|city)\s+in\s+(?:North\s+Vietnam|South\s+Vietnam|Vietnam|Việt\s+Nam)\b",
    r"\bI\s+was\s+born\s+in(?:\s+actually)?\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}\s+(?:Vietnam|Việt\s+Nam))\s+(?:18|19|20)\d{2}\b",
    r"\bborn\s+in\s+(North\s+Vietnam|South\s+Vietnam|Vietnam|Việt\s+Nam)\b",
    r"\bit(?:'s|\s+is)\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?,\s*[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?,\s*(?:Vietnam|Việt\s+Nam))\b",
    r"\btại\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?,\s*(?:Việt\s+Nam|Vietnam))\b",
    r"\b(?:I\s+was\s+)?born\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?,\s*(?:Vietnam|Việt\s+Nam|South\s+Vietnam|Laos|Cambodia))\b",
    r"\b(?:I\s+was\s+)?born\s+in\s+[A-ZÀ-Ỹa-zà-ỹ\s]{1,35}\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?\s+(?:Vietnam|Việt\s+Nam|South\s+Vietnam|Laos|Cambodia))\b",
    r"(?:(?:^|[.?!]\s*)(?:[A-Z][a-z]+\s+\d{1,2},\s+)?(?:18|19)\d{2}\s+in\s+)([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35}?,\s*(?:Vietnam|Việt\s+Nam|South\s+Vietnam|Laos|Cambodia))\b",
    r"\bI(?:'m|\s+am)\s+(?:originally\s+)?from\s+"
    r"([A-ZÀ-Ỹa-zà-ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{2,35}?)(?:,\s*(?:Vietnam|South\s+Vietnam|Việt\s+Nam|Laos|Cambodia)|\s+province|\s+city)",
    r"\bI\s+(?:grew\s+up|was\s+raised)\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{2,40}?)(?:\.|,|\s+(?:province|city|district|Vietnam|Laos|Cambodia))",
    r"\bfrom\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{2,30}?),\s*(?:Vietnam|South\s+Vietnam|Việt\s+Nam|North\s+Vietnam)\b",
    r"\bmy\s+hometown\s+(?:is|was)\s+([A-ZÀ-Ỹa-zà-ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{2,40}?)(?:\.|,|\s+in\s+Vietnam|\s+province)",
    r"\b(?:a\s+)?native\s+of\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s,]{2,35}?)(?:\.|,|\s+(?:province|city|Vietnam|Laos))",
    r"\b([A-Z][A-Za-z' -]{2,35},\s*[A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+)?)\b[^.?\n]{0,80}\bwhere\s+I\s+was\s+born\b",
]

_BIRTHPLACE_QUESTION_PATTERNS = [
    r"\bplace\s+of\s+birth\b",
    r"\bbirth\s*place\b",
    r"\bbirthplace\b",
    r"\bwhere\s+were\s+you\s+born\b",
    r"\bwhat\s+city\s+were\s+you\s+born(?:\s+in)?\b",
    r"\bwhere\s+you\s+were\s+born\b",
]

_BIRTHPLACE_DIRECT_ANSWER_PATTERNS = [
    r"^\s*(?:[A-Z][A-Za-zÀ-Ỹà-ỹ.' -]{1,24}:\s*)?(?:I\s+was\s+born\s+(?:in|at)\s+)?"
    r"([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35}(?:,\s*(?:Vietnam|Việt\s*Nam|South\s+Vietnam|North\s+Vietnam|Laos|Cambodia))?)\b",
    r"^\s*(?:[A-Z][A-Za-zÀ-Ỹà-ỹ.' -]{1,24}:\s*)?(?:I\s+was\s+born\s+(?:in|at)\s+)?"
    r"([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35}\s+(?:Vietnam|Việt\s*Nam|South\s+Vietnam|North\s+Vietnam|Laos|Cambodia))\b",
    r"^\s*(?:[A-Z][A-Za-zÀ-Ỹà-ỹ.' -]{1,24}:\s*)?(Vietnam|Việt\s*Nam|South\s+Vietnam|North\s+Vietnam|Laos|Cambodia)\b",
    r"^\s*(?:[A-Z][A-Za-zÀ-Ỹà-ỹ.' -]{1,24}:\s*)?(?:I\s+was\s+born\s+(?:in|at)\s+)?"
    r"([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{2,35})\b(?=\.|,|$)",
]

_BAD_BIRTHPLACE_VALUES = {
    "america",
    "california",
    "here",
    "huntington beach",
    "irvine",
    "linda vista",
    "los angeles",
    "my hometown",
    "orange county",
    "san diego",
    "santa ana",
    "santa monica",
    "southern california",
    "the city",
    "the countryside",
    "the village",
    "united states",
    "usa",
    "u.s.",
    "us",
}

_US_BIRTHPLACE_PATTERN = re.compile(
    r"\b(?:"
    r"United\s+States|U\.S\.A?\.?|USA|America|American|District\s+of\s+Columbia|Washington,\s*DC|"
    r"Alabama|Alaska|Arizona|Arkansas|California|Colorado|Connecticut|Delaware|Florida|Georgia|Hawaii|Idaho|Illinois|Indiana|Iowa|Kansas|Kentucky|Louisiana|Maine|Maryland|Massachusetts|Michigan|Minnesota|Mississippi|Missouri|Montana|Nebraska|Nevada|New\s+Hampshire|New\s+Jersey|New\s+Mexico|New\s+York|North\s+Carolina|North\s+Dakota|Ohio|Oklahoma|Oregon|Pennsylvania|Rhode\s+Island|South\s+Carolina|South\s+Dakota|Tennessee|Texas|Utah|Vermont|Virginia|Washington|West\s+Virginia|Wisconsin|Wyoming|"
    r"\bCA\b|\bTX\b|\bVA\b|\bMN\b|\bTN\b|\bNY\b|\bNJ\b|\bFL\b|\bWA\b|\bDC\b"
    r")\b",
    re.IGNORECASE,
)

_SUPP_LEAVE_YEAR_PATTERNS = [
    r"\b(?:left|fled|escaped\s+from|got\s+out\s+of|departed)\s+(?:Vietnam|Saigon|the\s+country|South\s+Vietnam)\s+(?:in\s+)?((?:19|20)\d{2})\b",
    r"\b(?:left|escaped\s+from)\s+Vietnam\s+(?:late|around|about)?\s*((?:19|20)\d{2})\b",
    r"\b(?:left|fled|escaped|evacuated)[^.?\n]{0,40}?Vietnam[^.?\n]{0,40}?\b(19[6-9][0-9]|20[0-2][0-9])\b",
    r"\b(?:left|fled|escaped|evacuated)\s+(?:Vietnam|Saigon|the\s+country|my\s+country|home)\b[^.?\n]{0,40}?\b(19[6-9][0-9]|20[0-2][0-9])\b",
    r"\b(?:left|fled|escaped|evacuated)\b[^.?\n]{0,30}?\bin\s+(?:\w+\s+)?(19[6-9][0-9]|20[0-2][0-9])\b",
    r"\bdeparted\b[^.?\n]{0,40}?\b(19[6-9][0-9]|20[0-2][0-9])\b",
    r"\b(?:I|we)\s+left\b[^.?\n]{0,30}?\b(19[6-9][0-9]|20[0-2][0-9])\b",
    r"\bapril\s+30(?:th)?[^.?\n]{0,20}?\b(1975)\b",
    r"\b(?:we|I)\s+(?:got\s+out|left)\s+(?:in\s+)?((?:19|20)\d{2})\b",
    r"\bI\s+leave\s+Vietnam\s+by\s+small\s+boat\W+((?:19|20)\d{2})\b",
    r"\bmy\s+time\s+to\s+escape\s+in\s+((?:19|20)\d{2})\b",
]

_AGE_STAGE_MAP = {
    "infant": 0,
    "baby": 0,
    "newborn": 0,
    "toddler": 2,
    "preschooler": 4,
    "child": 5,
    "kid": 5,
    "little": 5,
    "young": 7,
    "elementary": 8,
    "grade school": 8,
    "middle school": 12,
    "junior high": 12,
    "sixth grader": 11,
    "seventh grader": 12,
    "eighth grader": 13,
    "seventh grade": 12,
    "eighth grade": 13,
    "ninth grade": 14,
    "high school": 15,
    "teenager": 15,
    "teen": 15,
}

_SUPP_COUNTRY_PATTERNS = [
    (r"\b(?:born\s+in|from|grew\s+up\s+in)\b[^.?\n]{0,80}\b(Vietnam|Việt\s+Nam|South\s+Vietnam|North\s+Vietnam)\b", "Vietnam"),
    (r"\b(?:born\s+in|from|grew\s+up\s+in)\b[^.?\n]{0,80}\b(Laos)\b", "Laos"),
    (r"\b(?:born\s+in|from|grew\s+up\s+in)\b[^.?\n]{0,80}\b(Cambodia|Cambodian)\b", "Cambodia"),
    (r"\bSouth\s+Vietnam\b", "Vietnam"),
    (r"\bViệt\s+Nam\b", "Vietnam"),
]

_SUPP_COHORT_PATTERNS = [
    (r"\b(?:boat\s+people|by\s+boat|escape\s+by\s+boat|boat\s+escape)\b", "Boat people"),
    (r"\b(?:small\s+boat|fishing\s+boat|boat\s+trip|boat\s+person|pirates?)\b", "Boat people"),
    (r"\b(?:through|under)\s+(?:the\s+)?(?:ODP|Orderly\s+Departure\s+Program)\b", "ODP"),
    (r"\b(?:reunification\s+program|family\s+reunification)\b", "ODP"),
    (r"\bAmerasian\b", "Amerasian"),
    (r"\b(?:HO|Humanitarian\s+Operation)\b", "HO (Humanitarian Operation)"),
    (r"\bre[- ]education\s+camp\b", "HO (Humanitarian Operation)"),
    (r"\bpolitical\s+prisoner\b", "HO (Humanitarian Operation)"),
    (r"\b(?:land\s+escape|escaped\s+by\s+land)\b", "Land escape"),
    (r"\b(?:refugee\s+camp|camp\s+resettlement|resettled\s+from\s+(?:a\s+)?camp)\b", "Camp resettlement"),
    (r"\b(?:fall\s+of\s+Saigon|April\s+30(?:th)?(?:,?\s*1975)?|when\s+Saigon\s+fell)\b", "Wave 1 (Fall of Saigon)"),
    (r"\bevacuated?\b[^.?\n]{0,40}?\b1975\b", "Wave 1 (Fall of Saigon)"),
    (r"\b(?:USS\s+Midway|Camp\s+Pendleton|Guam|helicopter|airlift)\b", "Wave 1 (Fall of Saigon)"),
    (r"\bescaped?\s+by\s+boat\b", "Boat people"),
    (r"\b(?:took|on|boarded)\s+a\s+boat\b", "Boat people"),
]

_SUPP_PREMIG_OCC_PATTERNS = [
    r"\bI\s+was\s+(?:a\s+|an\s+)?([\w][\w\s\-/]{2,35}?)(?:\s+and\s+my\s+husband\s+had|\s+in\s+Vietnam|\.|,|$)",
    r"\bas\s+(?:a\s+)?([\w][\w\s\-/]{2,35}?\s+boy)\b",
    r"\bbefore\s+(?:I|we)\s+(?:came|left|arrived)[^.?\n]{0,80}\bI\s+(?:was|worked\s+as)\s+(?:a\s+|an\s+)?([\w][\w\s\-/]{2,35}?)(?:\.|,|\s+and\b|$)",
    r"\bin\s+Vietnam[^.?\n]{0,80}\bI\s+(?:was|worked\s+as)\s+(?:a\s+|an\s+)?([\w][\w\s\-/]{2,35}?)(?:\.|,|\s+and\b|$)",
    r"\bmy\s+(?:father|mother|parents?)\s+(?:was|were)\s+(?:a\s+|an\s+)?([\w][\w\s\-/]{2,35}?)(?:\.|,|\s+and\b|$)",
    r"\b(?:I|he|she)\s+worked\s+for\s+[^\n]{0,60}?\b(?:as\s+)?(?:a\s+|an\s+)?([\w][\w\s\-/]{2,35}?)(?:\.|,|\s+and\b|$)",
]

_NON_OCCUPATION_PHRASES = {
    "good boy", "young age", "small", "too small", "very small",
    "happy", "lucky", "poor", "very poor", "old", "very old",
}

_TRANSIT_LOCATIONS = (
    r"(?:refugee\s+)?camp|Malaysia|Mã\s+Lai|Thailand|Thái\s+Lan|"
    r"Philippines|Bataan|Palawan|Indonesia|Galang|Pulau\s+Bidong|"
    r"Songkhla|Hong\s+Kong|Hồng\s+Kông|transit"
)
_SUPP_TRANSIT_PATTERNS = [
    rf"\bspent\s+(\w+|\d{{1,2}})\s+(years?|months?)\s+(?:in|at)\s+(?:the\s+)?(?:{_TRANSIT_LOCATIONS})\b",
    rf"\b(?:I|we)\s+(?:was|were|stayed|lived|remained)\s+(?:in|at)\s+(?:the\s+)?(?:{_TRANSIT_LOCATIONS})\s+"
    rf"for\s+(?:about\s+|approximately\s+|almost\s+)?(\w+|\d{{1,2}})\s+(years?|months?)\b",
    rf"\bafter\s+(?:about\s+)?(\w+|\d{{1,2}})\s+(years?|months?)\s+(?:in|at)\s+(?:the\s+)?(?:{_TRANSIT_LOCATIONS})\b",
    rf"\b(\w+|\d{{1,2}})\s+(years?|months?)\s+in\s+(?:the\s+)?(?:{_TRANSIT_LOCATIONS})\b",
]
_TRANSIT_WORD_TO_NUM = {
    "one": 1, "a": 1, "an": 1, "two": 2, "three": 3, "four": 4,
    "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
}

# Vietnamese male middle-name markers beyond what step1 already knows
_VN_MALE_MIDDLE = {"văn", "van", "quốc", "quoc", "hữu", "huu", "minh"}
_VN_FEMALE_MIDDLE = {"thị", "thi", "ngọc", "ngoc", "thu", "thuy"}

# English given names not covered by step1's lists — used as a fallback
# when step1 name-based gender inference left Gender blank.
_SUPP_MALE_NAMES = {
    "tom", "tommy", "john", "hieu", "hugo", "roger", "charlie", "charlie", "stefan", "stephen",
    "alex", "tony", "larry", "larry", "leo", "leonard", "mike", "mikey",
    "rick", "ricky", "sam", "samuel", "ted", "tim", "timothy", "terry",
    "jerry", "barry", "gary", "larry", "harry", "marty", "danny", "donny",
    "randy", "andy", "sandy",  # sandy is ambiguous but leans male in Vietnamese-American context
    "kenny", "benny", "lenny", "denny", "johnny", "ronnie", "robbie",
    "freddie", "eddie", "georgie", "charlie",
    "scott", "brett", "troy", "kurt", "clint", "glen", "glen", "blaine",
    "brad", "chad", "shawn", "shaun", "sean", "shane", "brent", "lance",
    "grant", "blake", "cole", "cody", "travis", "tyler", "kyle",
    "derek", "darren", "darin", "bruce", "steve", "thien", "tam", "phat", "dan", "nam",
}

_SUPP_FEMALE_NAMES = {
    "debbie", "debby", "deb", "amy", "nicole", "nikki", "annie", "ann", "mimi",
    "mai", "mai-phuong", "tu-uyen", "uyen", "ysa", "suzie", "suzy", "susan", "hanh",
    "cathy", "kathy", "kathi", "tiffany", "tiff", "lien", "lian",
    "connie", "bonnie", "ronnie",  # ronnie female variant
    "tracey", "tracy", "stacy", "stacey", "marcy", "darcy",
    "shelley", "shelly", "kelly", "missy", "sissy", "cissy",
    "tammy", "sammy", "pam", "pamela", "jan", "janet", "janice",
    "cindy", "wendy", "brenda", "glenda", "linda",
    "cheryl", "carol", "carole", "carolyn", "marilyn",
    "beverly", "bev", "shirley", "sheila", "stella",
    "patty", "patti", "pat", "trish", "tricia", "tricia",
    "leann", "leanne", "joann", "joanne", "diann", "dianne",
    "kristy", "krissy", "betsy", "daisy", "katie", "kate", "paulette", "loan",
    "sandy",  # also in male above — handled by ambiguity → None
}
# Remove genuine ambiguous names from both to avoid wrong assignment
_AMBIGUOUS_NAMES = {"sandy", "ronnie", "sam", "pat", "terry", "tracy"}
_SUPP_MALE_NAMES   -= _AMBIGUOUS_NAMES
_SUPP_FEMALE_NAMES -= _AMBIGUOUS_NAMES

_SUPP_MALE_NAME_PHRASES = {
    "tu anh",
    "hieu nhu",
}


def _first_match(patterns: list, text: str) -> str | None:
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.MULTILINE)
        if m:
            return m.group(1) if m.lastindex else m.group(0)
    return None


def _normalise_birthplace(raw: str | None) -> str | None:
    if not raw:
        return None
    s = " ".join(str(raw).strip().split())
    s = re.sub(r"(?<=[A-Za-zÀ-Ỹà-ỹ])\d{1,2}\b", "", s)
    s = s.strip(" ,.;:-")
    s = re.sub(r"\bViệtNam\b", "Việt Nam", s, flags=re.IGNORECASE)
    s = re.sub(r"\bVietNam\b", "Vietnam", s, flags=re.IGNORECASE)
    s = re.sub(r"\bViet\s+Nam\b", "Vietnam", s, flags=re.IGNORECASE)
    s = re.sub(r"\bSouth\s*Viet\s*Nam\b", "South Vietnam", s, flags=re.IGNORECASE)
    s = re.sub(r"\bNorth\s*Viet\s*Nam\b", "North Vietnam", s, flags=re.IGNORECASE)
    s = re.sub(r"\b([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35})\s+(South Vietnam|North Vietnam)\b", r"\1, \2", s)
    s = re.sub(r"\b([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,35})\s+(Vietnam|Việt Nam|Laos|Cambodia)\b", r"\1, \2", s)
    s = re.sub(r",\s+(South|North),\s+Vietnam\b", r", \1 Vietnam", s)
    s = re.sub(r"\s{2,}", " ", s).strip(" ,.;:-")

    lower = s.lower()
    if lower in _BAD_BIRTHPLACE_VALUES:
        return None
    if re.search(r"\b(?:my\s+dad|my\s+father|my\s+mom|my\s+mother|my\s+parents?)\s+is\s+from\b", lower, re.IGNORECASE):
        return None
    if re.fullmatch(r"(?:mr|mrs|ms|miss)\.?", lower):
        return None
    if len(s) < 3:
        return None
    return s


def _extract_birthplace_from_qa(text: str) -> str | None:
    sample = re.sub(r"\s*\n\s*", " ", text[:12000])
    for qpat in _BIRTHPLACE_QUESTION_PATTERNS:
        for qm in re.finditer(qpat, sample, re.IGNORECASE):
            window = sample[qm.end():qm.end() + 220].lstrip(" ?:.-")
            direct = _first_match(_SUPP_BIRTHPLACE_PATTERNS, window)
            if direct:
                cleaned = _normalise_birthplace(direct)
                if cleaned:
                    return cleaned
            for apat in _BIRTHPLACE_DIRECT_ANSWER_PATTERNS:
                m = re.search(apat, window, re.IGNORECASE)
                if not m:
                    continue
                cleaned = _normalise_birthplace(m.group(1))
                if cleaned:
                    return cleaned
    return None


def _extract_birthplace_value(*texts: str) -> str | None:
    for text in texts:
        if not text:
            continue
        sample = re.sub(r"\s*\n\s*", " ", str(text))
        sample = re.sub(r"(?<=[A-Za-zÀ-Ỹà-ỹ])\d{1,2}\b", "", sample)
        sample = re.sub(r"\s{2,}", " ", sample).strip()

        direct = _first_match(_SUPP_BIRTHPLACE_PATTERNS, sample)
        if direct:
            cleaned = _normalise_birthplace(direct)
            if cleaned:
                return cleaned

        qa = _extract_birthplace_from_qa(sample)
        if qa:
            return qa
    return None


def _is_us_birthplace(value: object) -> bool:
    if value is None or pd.isna(value):
        return False
    return bool(_US_BIRTHPLACE_PATTERN.search(str(value)))


def _infer_transit_arrival(text: str, leave_year: int) -> int | None:
    sample = re.sub(r"\s*\n\s*", " ", text[:8000])
    for pat in _SUPP_TRANSIT_PATTERNS:
        m = re.search(pat, sample, re.IGNORECASE)
        if m:
            raw_n = m.group(1)
            unit  = m.group(2).lower()
            n = _TRANSIT_WORD_TO_NUM.get(raw_n.lower())
            if n is None:
                try:
                    n = int(raw_n)
                except ValueError:
                    continue
            transit_years = n if "year" in unit else max(1, round(n / 12))
            yr = leave_year + transit_years
            if 1950 <= yr <= 2010:
                return yr
    return None


def _extract_age_at_arrival_phrase(text: str) -> int | None:
    sample = re.sub(r"\s*\n\s*", " ", text[:12000])
    patterns = [
        r"\b(?:came|arrived|got\s+here|moved\s+here)\s+(?:when\s+I\s+was\s+(?:only\s+|about\s+|around\s+)?)?(\d{1,2})(?:\s+years?\s+old)?\b",
        r"\bI\s+was\s+(?:only\s+|about\s+|around\s+)?(\d{1,2})(?:\s+years?\s+old)?\s+when\s+(?:we|I)\s+(?:came|arrived|got\s+here|moved\s+here)\b",
        r"\bI\s+was\s+in\s+Vietnam\s+until\s+(\d{1,2})\b",
        r"\b(?:from\s+the\s+day\s+I\s+was\s+born\s+until|until)\s+(\d{1,2})\b[^.?\n]{0,120}\b(?:grew\s+up\s+in\s+the\s+U\.?S|in\s+the\s+U\.?S|came\s+to\s+the\s+U\.?S|came\s+to\s+America)\b",
        r"\bbefore\s+I\s+got\s+into\s+the\s+United\s+States[^.?\n]{0,100}\b(?:when\s+I\s+was\s+)?(\d{1,2})\b",
        r"\bfrom\s+when\s+I\s+was\s+\d{1,2}\s+to\s+(\d{1,2})\s*-\s*before\s+I\s+got\s+into\s+the\s+United\s+States\b",
        r"\bI\s+came\s+(?:here|to\s+the\s+United\s+States|to\s+America)\s+(?:when\s+I\s+was\s+about\s+|when\s+I\s+was\s+)?(\d{1,2})(?:\s+and\s+a\s+half)?\b",
        r"\bI\s+came\s+I\s+was\s+only\s+(\d{1,2})\b",
    ]
    for pat in patterns:
        m = re.search(pat, sample, re.IGNORECASE)
        if not m:
            continue
        try:
            age = int(m.group(1))
        except Exception:
            continue
        if 0 <= age <= 80:
            return age

    # Maturity/life-stage phrasing when no numeric age is stated.
    stage_patterns = [
        r"\b(?:came|arrived|got\s+here|moved\s+here)\b[^.?\n]{0,40}\b(as\s+an?\s+|as\s+a\s+)?(infant|baby|newborn|toddler|preschooler|child|kid|little|young|elementary|grade\s+school|middle\s+school|junior\s+high|high\s+school|teenager|teen)\b",
        r"\bI\s+was\s+(?:just\s+|still\s+|only\s+)?(an?\s+)?(infant|baby|newborn|toddler|preschooler|child|kid|little|young|elementary|grade\s+school|middle\s+school|junior\s+high|high\s+school|teenager|teen)\s+when\s+(?:we|I)\s+(?:came|arrived|got\s+here|moved\s+here)\b",
        r"\bwhen\s+I\s+came\b[^.?\n]{0,40}\bI\s+was\s+(?:in\s+)?(middle\s+school|junior\s+high|high\s+school|sixth\s+grader|seventh\s+grader|eighth\s+grader|teenager|teen)\b",
        r"\bwhen\s+I\s+came\s+to\s+America\b[^.?\n]{0,60}\bstarted\s+(seventh\s+grade|eighth\s+grade|ninth\s+grade)\b",
    ]
    for pat in stage_patterns:
        m = re.search(pat, sample, re.IGNORECASE)
        if not m:
            continue
        stage = next((g for g in m.groups() if g and g.strip().lower() in _AGE_STAGE_MAP), None)
        if stage:
            return _AGE_STAGE_MAP[stage.strip().lower()]
    return None


def _infer_arrival_year_from_age_phrase(text: str, birth_year: object) -> int | None:
    if pd.isna(birth_year):
        return None
    age = _extract_age_at_arrival_phrase(text)
    if age is not None:
        try:
            yr = int(birth_year) + age
        except Exception:
            return None
        if 1950 <= yr <= 2010:
            return yr
    return None


def _derive_birth_year_from_age_now(text: str, interview_year: object) -> int | None:
    """Back-calculate birth year from present-age statements when interview year is known."""
    if pd.isna(interview_year):
        return None

    sample = str(text or "")
    sample = sample.replace("’", "'").replace("‘", "'").replace("`", "'")
    sample = re.sub(r"\s*\n\s*", " ", sample)
    sample = re.sub(r"\s+", " ", sample)
    sample = re.sub(r"\bI\s*'\s*m\b", "I'm", sample, flags=re.IGNORECASE)
    sample = re.sub(r"\bI\s*[’']\s*m\b", "I'm", sample, flags=re.IGNORECASE)
    sample = re.sub(r"\bI\s+m\b", "I'm", sample, flags=re.IGNORECASE)

    patterns = [
        r"\b(\d{1,3})\s*tuổi\b",
        r"\bat\s+the\s+time\s+of\s+(?:the\s+)?interview[^.?\n]{0,30}\b(\d{1,3})\s+years?\s+old\b",
        r"\bage\s*[:\-]?\s*(\d{1,3})\b",
        r"\b(\d{1,3})-year-old\b",
        r"\bI(?:'m|\s+am)\s+currently\s+am\s+(\d{1,3})\b",
        r"\bcurrently\s+I(?:'m|\s+am)\s+(\d{1,3})\b",
        r"\bcurrently\s+am\s+(\d{1,3})\b",
        r"\bI(?:'m|\s+am)\s+(\d{1,3})\s+years?\s+old\s+now\b",
        r"\bI(?:'m|\s+am)\s+(\d{1,3})\s+years?\s+old\b",
        r"\bcurrently[^.?\n]{0,20}\b(\d{1,3})\s+years?\s+old\b",
    ]
    for pat in patterns:
        m = re.search(pat, sample, re.IGNORECASE)
        if not m:
            continue
        try:
            age = int(m.group(1))
            yr = int(interview_year) - age
        except Exception:
            continue
        if 1880 <= yr <= 2000:
            return yr
    return None


def _derive_birth_year_from_past_age_event(text: str) -> int | None:
    """
    Back-calculate birth year from retrospective age statements tied to a
    concrete year, e.g.:
      - "I was back in 1998 ... I was 18 years old when we went back"
      - "In 1979 ... I was six years old"
    """
    sample = re.sub(r"\s*\n\s*", " ", text[:20000])
    patterns = [
        r"\bback\s+in\s+((?:19|20)\d{2})[^.?\n]{0,220}\bI\s+(?:was|am)\s+(\d{1,2})\s+years?\s+old\b",
        r"\bin\s+((?:19|20)\d{2})[^.?\n]{0,220}\bI\s+(?:was|am)\s+(\d{1,2})\s+years?\s+old\b",
        r"\bI\s+(?:was|am)\s+(\d{1,2})\s+years?\s+old[^.?\n]{0,220}\bback\s+in\s+((?:19|20)\d{2})\b",
        r"\bI\s+(?:was|am)\s+(\d{1,2})\s+years?\s+old[^.?\n]{0,220}\bin\s+((?:19|20)\d{2})\b",
    ]
    for pat in patterns:
        m = re.search(pat, sample, re.IGNORECASE)
        if not m:
            continue
        nums = [int(g) for g in m.groups()]
        event_year = max(nums)
        age = min(nums)
        birth_year = event_year - age
        if 1880 <= birth_year <= 2000:
            return birth_year
    return None


def _coerce_interview_year(interview_year: object, interview_date: object, header_text: str) -> int | None:
    if pd.notna(interview_year):
        try:
            year = int(float(interview_year))
        except Exception:
            year = None
        if year is not None and 1950 <= year <= 2030:
            return year
    sample = " ".join(str(x or "") for x in [interview_date, header_text])
    years = [int(y) for y in re.findall(r"\b(19[5-9]\d|20[0-2]\d)\b", sample)]
    if years:
        return max(years)
    return None


def _extract_age_at_departure_phrase(text: str) -> int | None:
    sample = re.sub(r"\s*\n\s*", " ", text[:12000])
    patterns = [
        r"\bleft\s+Vietnam\s+in\s+(?:18|19|20)\d{2}\s+at\s+the\s+age\s+(?:of\s+)?(\d{1,2})\b",
        r"\b(?:left|we\s+left|I\s+left)[^.?\n]{0,80}\b(?:when\s+I\s+was\s+)?(\d{1,2})\s+years?\s+old\b",
        r"\b(?:took\s+the\s+boat\s+trip\s+to\s+escape|escaped?|went\s+by\s+boat)\s+at\s+(\d{1,2})\s+years?\s+old\b",
        r"\b(?:I\s+was|I(?:'m|\s+am))\s+(\d{1,2})\s+years?\s+old[^.?\n]{0,80}\b(?:when\s+we\s+left|when\s+I\s+left|before\s+1975|in\s+1975)\b",
        r"\b((?:19|20)\d{2})\s+was\s+when\s+we\s+left[^.?\n]{0,60}\bI\s+was\s+(\d{1,2})\s+years?\s+old\b",
    ]
    for pat in patterns:
        m = re.search(pat, sample, re.IGNORECASE)
        if not m:
            continue
        age_group = m.group(m.lastindex)
        try:
            age = int(age_group)
        except Exception:
            continue
        if 0 <= age <= 80:
            return age
    return None


def _normalize_birth_year_value(raw_year: str | int | None) -> int | None:
    """
    Normalize regex-captured birth years.
    Handles both 4-digit years and two-digit DOB shorthand like '56' -> 1956.
    """
    if raw_year is None or (isinstance(raw_year, float) and pd.isna(raw_year)):
        return None
    try:
        year = int(str(raw_year).strip())
    except Exception:
        return None

    # Upper bound 2000: QC also flags by > 2000 as implausible; align here
    # so values in 2001-2005 are never set only to be immediately QC-flagged.
    if 1880 <= year <= 2000:
        return year
    if 0 <= year <= 99:
        inferred = 1900 + year
        if 1880 <= inferred <= 2000:
            return inferred
    return None


def _infer_gender_extra(name: str | None) -> str | None:
    """
    Extra gender inference not in step1:
    1. Vietnamese male middle-name markers (Quốc, Hữu …)
    2. English given-name lookup from supplementary lists
    """
    if not name:
        return None
    full_name = " ".join(name.strip().lower().replace("-", " ").split())
    first_two = " ".join(full_name.split()[:2])
    if full_name in _SUPP_MALE_NAME_PHRASES or first_two in _SUPP_MALE_NAME_PHRASES:
        return "Male"
    tokens = []
    for raw in name.strip().split():
        raw = raw.strip(".,;:").lower()
        if not raw:
            continue
        tokens.append(raw)
        tokens.extend(part for part in raw.split("-") if part)

    # Vietnamese marker tokens can appear in middle position or, in some
    # English-order names, at the beginning.
    for tok in tokens[:-1]:
        if tok in _VN_MALE_MIDDLE:
            return "Male"
        if tok in _VN_FEMALE_MIDDLE:
            return "Female"

    # English given-name lookup across the first token and any hyphen-split parts.
    probe_tokens = []
    if tokens:
        probe_tokens.append(tokens[0])
    probe_tokens.extend(tokens[:3])
    for tok in probe_tokens:
        if tok in _SUPP_MALE_NAMES:
            return "Male"
        if tok in _SUPP_FEMALE_NAMES:
            return "Female"

    return None


def _infer_gender_from_header(header_text: str | None) -> str | None:
    if not header_text:
        return None
    if re.search(r"\b(?:Mr\.?|Mister)\s+[A-ZÀ-Ỹ]", header_text, re.IGNORECASE):
        return "Male"
    if re.search(r"\b(?:Ms\.?|Mrs\.?|Miss)\s+[A-ZÀ-Ỹ]", header_text, re.IGNORECASE):
        return "Female"
    return None


def _infer_gender_from_pronouns(full_text: str, narrator_name: str | None) -> str | None:
    """
    Scan the full transcript (including interviewer turns) for third-person
    pronoun references.  Uses a short window around the narrator's name to
    avoid counting unrelated pronouns.

    Returns 'Male', 'Female', or None if the signal is too weak.
    """
    if not full_text:
        return None

    text_lower = full_text[:15000].lower()

    # If narrator name is known, look in a ±300-char window around each mention.
    # Companion files often use short references like "Mr. Lanh" or "Miss Mimi",
    # so also search around the final token of the name.
    windows = []
    if narrator_name:
        parts = [p.lower() for p in narrator_name.strip().split() if p]
        probes = []
        if parts:
            probes.append(parts[0])
            if len(parts) > 1:
                probes.append(parts[-1])
        for probe in probes:
            for m in re.finditer(re.escape(probe), text_lower):
                start = max(0, m.start() - 300)
                end   = min(len(text_lower), m.end() + 300)
                windows.append(text_lower[start:end])
    if not windows:
        windows = [text_lower]

    combined = " ".join(windows)
    she_count = len(re.findall(r'\b(she|her|herself|hers)\b', combined))
    he_count  = len(re.findall(r'\b(he|him|his|himself)\b', combined))
    global_she = len(re.findall(r'\b(she|her|herself|hers)\b', text_lower))
    global_he  = len(re.findall(r'\b(he|him|his|himself)\b', text_lower))

    threshold = 2   # need at least this many hits
    ratio     = 2.5 # dominant gender needs this multiple of the other

    if she_count >= threshold and she_count >= he_count * ratio:
        return "Female"
    if he_count >= threshold and he_count >= she_count * ratio:
        return "Male"
    # Fallback for support files like time logs that summarize the narrator as
    # "He ..." or "She ..." without repeating the full name nearby.
    if global_she >= 3 and global_she >= global_he * ratio:
        return "Female"
    if global_he >= 3 and global_he >= global_she * ratio:
        return "Male"
    return None


def _resolve_gender(narrator_name: str | None, gender_context_text: str) -> str | None:
    """
    Prefer strong textual evidence (honorifics/pronouns) over name heuristics.
    This avoids rows where a Vietnamese marker like 'Minh' incorrectly overrides
    clear 'she/her' support text.
    """
    g_header = _infer_gender_from_header(gender_context_text)
    g_pron = _infer_gender_from_pronouns(gender_context_text, narrator_name)
    g_name = _infer_gender_extra(narrator_name)
    if g_header:
        return g_header
    if g_pron:
        return g_pron
    return g_name


# ---------------------------------------------------------------------------

## Block 4 · Supplementary Extraction — Single-pass All Fields

One PDF read per narrator: extracts all remaining variables in a single pass.

In [40]:
# 1b. Supplementary extraction — single-pass, all fields, one PDF read per narrator
# ---------------------------------------------------------------------------


def supplementary_extraction_all(df: pd.DataFrame) -> pd.DataFrame:
    """
    Single-pass supplementary extraction for all study variables.

    For every narrator that is still missing ANY key field after step1,
    the transcript text already saved in raw_transcripts.csv is used once and
    all patterns are applied in that single pass.

    Fields filled:
      Core       — Interviewer, Birth_Year, Year_Left_Country, Year_of_Arrival,
                   Birthplace, Country_of_Origin, Gender, Refugee_Cohort
      Education  — Highest_Education

    Uses narrator_dialogue as the primary text source for extraction.
    Supporting context is read in this order: life map, time log, field notes,
    then the Step 1 split transcript columns.
    """
    ALL_FIELDS = [
        "Birth_Year", "Year_Left_Country", "Year_of_Arrival", "Birthplace",
        "Country_of_Origin", "Gender", "Refugee_Cohort", "Interviewer",
        "Highest_Education",
    ]

    # Ensure Year_of_Arrival_Inferred column exists before we write to it
    if "Year_of_Arrival_Inferred" not in df.columns:
        df = df.copy()
        df["Year_of_Arrival_Inferred"] = None

    # Determine which rows still need work
    present = [f for f in ALL_FIELDS if f in df.columns]
    needs_work = df[present].isna().any(axis=1) if present else pd.Series(False, index=df.index)
    n_rows = int(needs_work.sum())
    if n_rows == 0:
        print("\nSupplementary extraction: all fields complete — nothing to fill.")
        return df

    print(f"\n{'='*60}")
    print(f"  SUPPLEMENTARY EXTRACTION  ({n_rows} narrators have gaps)")
    print(f"{'='*60}")

    # Coverage before this pass
    for f in ALL_FIELDS:
        n_have = int(df[f].notna().sum()) if f in df.columns else 0
        print(f"  {f:<28} before: {n_have:3d}/{len(df)}")
    print()

    df = prepare_clean_text_columns(df.copy())
    recovered = {f: 0 for f in ALL_FIELDS}

    pre_candidates = [
        "narrator_dialogue_pre_us_family_history",
        "pre_us_family_history",
        "narrator_dialogue_pre_america",
        "pre_america",
    ]
    post_candidates = [
        "narrator_dialogue_post_us_life",
        "post_us_life",
        "narrator_dialogue_post_america",
        "post_america",
    ]
    pre_col = next((c for c in pre_candidates if c in df.columns), None)
    post_col = next((c for c in post_candidates if c in df.columns), None)
    has_pre_col = pre_col is not None
    has_post_col = post_col is not None

    for idx, row in df[needs_work].iterrows():
        # ── Get narrator text from CSV only ────────────────────────────────
        narrator_text = str(row.get("narrator_dialogue_clean") or row.get("narrator_dialogue") or "").strip()
        life_map_text = str(row.get("life_map_text_clean") or row.get("life_map_text") or row.get("Life_Map_Text") or "").strip()
        time_log_text = str(row.get("time_log_text_clean") or row.get("time_log_text") or row.get("Time_Log_Text") or "").strip()
        field_notes_text = str(row.get("field_notes_text_clean") or row.get("field_notes_text") or row.get("Field_Notes_Text") or "").strip()
        pre_context = ""
        post_context = ""
        if has_pre_col:
            pre_context = str(
                row.get("narrator_dialogue_pre_us_family_history_clean")
                or row.get(pre_col)
                or ""
            ).strip()
        if has_post_col:
            post_context = str(
                row.get("narrator_dialogue_post_us_life_clean")
                or row.get(post_col)
                or ""
            ).strip()
        context_text = " ".join(
            x for x in [
                life_map_text,
                time_log_text,
                field_notes_text,
                pre_context,
                post_context,
            ] if x
        ).strip()
        if not narrator_text:
            narrator_text = context_text
        header = str(row.get("header_text_clean") or row.get("header_text") or "")
        arrival_context_text = " ".join(
            x for x in [
                life_map_text,
                narrator_text,
                time_log_text,
                field_notes_text,
                pre_context,
                post_context,
                header,
            ] if x
        ).strip()
        gender_context_text = " ".join(
            x for x in [
                header,
                narrator_text,
                life_map_text,
                time_log_text,
                field_notes_text,
            ] if x
        ).strip()

        flat          = re.sub(r"\s*\n\s*", " ", narrator_text[:10000])
        narrator_name = row.get("Narrator_Name", "")

        # ── Interviewer ────────────────────────────────────────────────────
        if pd.isna(row.get("Interviewer")):
            if header:
                m = re.search(r"(?im)^\s*Interview\s*[:\-]\s*(.+)$", header)
                if m:
                    val = m.group(1).strip()
                    if val:
                        df.at[idx, "Interviewer"] = val
                        recovered["Interviewer"] += 1

        # ── Year_of_Arrival ────────────────────────────────────────────────
        if pd.isna(row.get("Year_of_Arrival")):
            filled = False

            # 1. Fixed-year phrases (Fall of Saigon, April 30, etc.)
            arrival_search_text = re.sub(r"\s*\n\s*", " ", arrival_context_text[:60000]) if arrival_context_text else flat

            for pat, implied_yr in _SUPP_ARRIVAL_FIXED:
                if re.search(pat, arrival_search_text, re.IGNORECASE):
                    df.at[idx, "Year_of_Arrival"] = implied_yr
                    df.at[idx, "Year_of_Arrival_Inferred"] = 2
                    recovered["Year_of_Arrival"] += 1
                    filled = True
                    break

            # 2. Direct numeric patterns
            if not filled:
                val = _first_match(_SUPP_ARRIVAL_PATTERNS, arrival_search_text)
                if val and 1950 <= int(val) <= 2010:
                    df.at[idx, "Year_of_Arrival"] = int(val)
                    recovered["Year_of_Arrival"] += 1
                    filled = True

            # 3. Transit-duration inference (Year_Left + camp stay)
            if not filled and pd.notna(row.get("Year_Left_Country")):
                yr = _infer_transit_arrival(arrival_context_text or narrator_text, int(row["Year_Left_Country"]))
                if yr:
                    df.at[idx, "Year_of_Arrival"] = yr
                    df.at[idx, "Year_of_Arrival_Inferred"] = 2
                    recovered["Year_of_Arrival"] += 1
                    filled = True

            # 4. Year_Left = 1975 → assign 1975 directly (Fall-of-Saigon cohort)
            if not filled and pd.notna(row.get("Year_Left_Country")):
                leave_yr = int(row["Year_Left_Country"])
                if leave_yr == 1975:
                    df.at[idx, "Year_of_Arrival"] = 1975
                    df.at[idx, "Year_of_Arrival_Inferred"] = 2
                    recovered["Year_of_Arrival"] += 1
                    filled = True

            # 5. Soft estimate: Year_Left + 1 (typical 1-year transit lag)
            if not filled and pd.notna(row.get("Year_Left_Country")):
                leave_yr = int(row["Year_Left_Country"])
                est_yr   = leave_yr + 1
                if 1950 <= est_yr <= 2010:
                    df.at[idx, "Year_of_Arrival"] = est_yr
                    df.at[idx, "Year_of_Arrival_Inferred"] = 3
                    recovered["Year_of_Arrival"] += 1

            # 6. Age-at-arrival phrasing when birth year is known
            if not filled and pd.notna(row.get("Birth_Year")):
                yr = _infer_arrival_year_from_age_phrase(arrival_context_text, row.get("Birth_Year"))
                if yr:
                    df.at[idx, "Year_of_Arrival"] = yr
                    df.at[idx, "Year_of_Arrival_Inferred"] = 2
                    recovered["Year_of_Arrival"] += 1
                    filled = True

        # ── Year_Left_Country ──────────────────────────────────────────────
        if pd.isna(row.get("Year_Left_Country")):
            leave_val = _first_match(_SUPP_LEAVE_YEAR_PATTERNS, flat)
            if leave_val and 1950 <= int(leave_val) <= 2010:
                df.at[idx, "Year_Left_Country"] = int(leave_val)
                recovered["Year_Left_Country"] += 1
            elif re.search(r"\b(?:after\s+(?:the\s+)?fall\s+of\s+Saigon|April\s+30(?:th)?,?\s*1975|when\s+Saigon\s+fell)\b", flat, re.IGNORECASE):
                df.at[idx, "Year_Left_Country"] = 1975
                recovered["Year_Left_Country"] += 1

        # ── Birth_Year ─────────────────────────────────────────────────────
        if pd.isna(row.get("Birth_Year")):
            birth_search_text = " ".join(
                x for x in [
                    life_map_text,
                    header,
                    narrator_text,
                    time_log_text,
                    field_notes_text,
                    pre_context,
                    post_context,
                ]
                if x
            ).strip()
            # OCR/transcript footnote digits like "Can Tho2" or "Tam1" can
            # break otherwise-correct date/place patterns.
            birth_search_text = re.sub(r"(?<=[A-Za-zÀ-Ỹà-ỹ])\d{1,2}\b", "", birth_search_text)
            birth_search_text = re.sub(r"\s{2,}", " ", birth_search_text)
            norm_birth_year = None
            for pat in _SUPP_BIRTH_YEAR_PATTERNS:
                m = re.search(pat, birth_search_text, re.IGNORECASE)
                if not m:
                    continue
                val = m.group(1) if m.lastindex else m.group(0)
                norm_birth_year = _normalize_birth_year_value(val)
                if norm_birth_year is not None:
                    break
            if norm_birth_year is not None:
                df.at[idx, "Birth_Year"] = norm_birth_year
                recovered["Birth_Year"] += 1
            else:
                current_interview_year = _coerce_interview_year(
                    row.get("Interview_Year"),
                    row.get("Interview_Date"),
                    header,
                )
                yr = _derive_birth_year_from_age_now(birth_search_text, current_interview_year)
                if yr:
                    df.at[idx, "Birth_Year"] = yr
                    recovered["Birth_Year"] += 1
                else:
                    yr = _derive_birth_year_from_past_age_event(birth_search_text)
                    if yr:
                        df.at[idx, "Birth_Year"] = yr
                        recovered["Birth_Year"] += 1
                        continue
                    current_arrival_year = df.at[idx, "Year_of_Arrival"]
                    current_leave_year = df.at[idx, "Year_Left_Country"]
                    if pd.notna(current_arrival_year):
                        age_at_arrival = _extract_age_at_arrival_phrase(arrival_context_text)
                        if age_at_arrival is not None:
                            try:
                                yr = int(float(current_arrival_year)) - int(age_at_arrival)
                            except Exception:
                                yr = None
                            if yr is not None and 1880 <= yr <= 2000:
                                df.at[idx, "Birth_Year"] = yr
                                recovered["Birth_Year"] += 1
                    elif pd.notna(current_leave_year):
                        age_at_departure = _extract_age_at_departure_phrase(arrival_context_text)
                        if age_at_departure is not None:
                            try:
                                yr = int(float(current_leave_year)) - int(age_at_departure)
                            except Exception:
                                yr = None
                            if yr is not None and 1880 <= yr <= 2000:
                                df.at[idx, "Birth_Year"] = yr
                                recovered["Birth_Year"] += 1

            # Final conservative row-level fallbacks after close reading.
            if pd.isna(df.at[idx, "Birth_Year"]):
                interview_id = str(row.get("Interview_ID", "")).strip()
                if interview_id == "VAOHP0038.2_F01":
                    # "I was born here" and "I was back in 1998 ... I was 18 years old"
                    # imply a birth year of 1980.
                    df.at[idx, "Birth_Year"] = 1980
                    recovered["Birth_Year"] += 1
                elif interview_id == "VAOHP0027":
                    # Transcript Q&A gives DOB as 11/28/81.
                    df.at[idx, "Birth_Year"] = 1981
                    recovered["Birth_Year"] += 1
                elif interview_id == "VAOHP0123_F01":
                    # Transcript states "date of birth is December 1980".
                    df.at[idx, "Birth_Year"] = 1980
                    recovered["Birth_Year"] += 1
                elif interview_id == "VAOHP0188":
                    # Transcript Q&A gives DOB as April 10th, 1966.
                    df.at[idx, "Birth_Year"] = 1966
                    recovered["Birth_Year"] += 1
                elif interview_id == "VAHF0006":
                    # Interview date is November 5, 2010 and narrator states age 24.
                    # Conservative inference: he had already turned 24 in 2010.
                    df.at[idx, "Birth_Year"] = 1986
                    recovered["Birth_Year"] += 1
                elif interview_id == "VAOHP0377":
                    # Interview date is February 21, 2019 and narrator states age 61.
                    # Conservative arithmetic gives a likely birth year of 1958.
                    df.at[idx, "Birth_Year"] = 1958
                    recovered["Birth_Year"] += 1
                elif interview_id == "VAOHP379":
                    # Header says interview was in 2019 and narrator states "56 tuổi".
                    df.at[idx, "Birth_Year"] = 1963
                    recovered["Birth_Year"] += 1

        # ── Birthplace ─────────────────────────────────────────────────────
        if pd.isna(row.get("Birthplace")):
            candidate = _extract_birthplace_value(
                header,
                life_map_text,
                narrator_text,
                time_log_text,
                field_notes_text,
                flat,
            )
            if candidate:
                df.at[idx, "Birthplace"] = candidate
                recovered["Birthplace"] += 1
            else:
                interview_id = str(row.get("Interview_ID", "")).strip()
                # Conservative row-level fallbacks where the transcript states
                # birthplace directly but OCR/disfluency makes generic matching brittle.
                if interview_id == "VAHF0006" and re.search(r"\bAmarillo,\s+Texas\b[^.?\n]{0,80}\bwhere\s+I\s+was\s+born\b", flat, re.IGNORECASE):
                    df.at[idx, "Birthplace"] = "Amarillo, Texas"
                    recovered["Birthplace"] += 1

        # ── Country_of_Origin ──────────────────────────────────────────────
        if pd.isna(row.get("Country_of_Origin")) or row.get("Country_of_Origin") == "Unknown":
            for pat, country in _SUPP_COUNTRY_PATTERNS:
                if re.search(pat, flat, re.IGNORECASE):
                    df.at[idx, "Country_of_Origin"] = country
                    recovered["Country_of_Origin"] += 1
                    break

        # ── Gender ─────────────────────────────────────────────────────────
        if pd.isna(row.get("Gender")):
            g = _resolve_gender(narrator_name, gender_context_text)
            if g:
                df.at[idx, "Gender"] = g
                recovered["Gender"] += 1

        # ── Refugee_Cohort ─────────────────────────────────────────────────
        if pd.isna(row.get("Refugee_Cohort")):
            cohort_search_text = " ".join(
                x for x in [life_map_text, time_log_text, field_notes_text, header, narrator_text, pre_context, post_context]
                if x
            )
            for pat, cohort in _SUPP_COHORT_PATTERNS:
                if re.search(pat, cohort_search_text, re.IGNORECASE):
                    df.at[idx, "Refugee_Cohort"] = cohort
                    recovered["Refugee_Cohort"] += 1
                    break

        # ── Highest_Education ──────────────────────────────────────────────
        if pd.isna(row.get("Highest_Education")):
            interview_id = str(row.get("Interview_ID", "")).strip()
            education_search_text = " ".join(
                x for x in [life_map_text, time_log_text, field_notes_text, header, narrator_text]
                if x
            )
            val = _first_match(_SUPP_EDUCATION_PATTERNS, education_search_text)
            if val:
                level = _normalise_education(val)
                if level:
                    df.at[idx, "Highest_Education"] = level
                    recovered["Highest_Education"] += 1
            elif interview_id in {
                "VAOHP0015_F01",
                "VAOHP0113",
                "VAOHP0116",
                "VAOHP0170",
                "VAOHP0249",
                "VAOHP0252",
                "VAOHP035",
                "VAOHP0057_F01_Eng",
            }:
                # Conservative row-level fallbacks from close transcript reading.
                if interview_id == "VAOHP0015_F01":
                    # Coleman College degree in Computer Information Science.
                    df.at[idx, "Highest_Education"] = "BA"
                elif interview_id == "VAOHP0113":
                    # Engineering school in Switzerland and master's in London.
                    df.at[idx, "Highest_Education"] = "MA/MS"
                elif interview_id == "VAOHP0116":
                    # States she studied medicine and got a medical degree.
                    df.at[idx, "Highest_Education"] = "Doctorate/Professional"
                elif interview_id == "VAOHP0170":
                    # Time log notes Cal State Long Beach and then a master's.
                    df.at[idx, "Highest_Education"] = "MA/MS"
                elif interview_id == "VAOHP0249":
                    # Life map says national technical school / machine shop and graduated.
                    df.at[idx, "Highest_Education"] = "Vocational/Trade"
                elif interview_id == "VAOHP0252":
                    # Life map says degree in chemical engineering.
                    df.at[idx, "Highest_Education"] = "BS"
                elif interview_id == "VAOHP035":
                    # Life map explicitly says PhD in Electrical Engineering at UCLA.
                    df.at[idx, "Highest_Education"] = "PhD"
                elif interview_id == "VAOHP0057_F01_Eng":
                    # Narrator says she did not get to go to school at all.
                    df.at[idx, "Highest_Education"] = "No formal schooling"
                recovered["Highest_Education"] += 1
            elif interview_id == "VAOHP0077":
                # Close read shows medical/medic experience but no clear terminal
                # schooling level, so mark it explicitly as unavailable.
                df.at[idx, "Highest_Education"] = "N/A"

    print("  Recovered this pass:")
    for field, cnt in recovered.items():
        total = int(df[field].notna().sum()) if field in df.columns else 0
        print(f"    {field:<28} +{cnt:<4}  total filled: {total}/{len(df)}")
    return df


# ---------------------------------------------------------------------------

## Block 5 · Self-ID & Occupational Patterns

Pattern dictionaries for present-state self-identification phrases and occupational tier mapping.

In [41]:
# (Kept for external callers — pattern dicts and helpers below)
# ---------------------------------------------------------------------------

# Present-state / self-identification phrases
_SUPP_FINAL_OCC_PATTERNS = [
    # "I work as a / I've been working as a ..."
    r"\bI(?:'ve|\s+have)?\s+(?:been\s+)?work(?:ing|ed)\s+as\s+(?:a\s+|an\s+)?([\w][\w\s\-/]{2,40}?)(?:\s+for\s+\d|\s+since\b|\s+at\b|\s+in\b|\.|,|$)",
    # "I am / I'm a ... now/these days/today"
    r"\bI(?:'m|\s+am)\s+(?:a\s+|an\s+)([\w][\w\s\-/]{2,40}?)\s+(?:now|today|these\s+days|at\s+(?:this\s+)?(?:point|time|moment))\b",
    # "now I am / now I work as ..."
    r"\bnow\s+I\s+(?:am\s+(?:a\s+|an\s+)|work(?:ing)?\s+as\s+(?:a\s+|an\s+)?)([\w][\w\s\-/]{2,40}?)(?:\.|,|\s+and\b|$)",
    # "I retired as / I retired from ..."
    r"\bI\s+(?:have\s+)?retired\s+(?:as\s+(?:a\s+|an\s+)|from\s+(?:my\s+(?:job|career|position|work)\s+as\s+(?:a\s+|an\s+))?)([\w][\w\s\-/]{2,40}?)(?:\.|,|\s+after\b|\s+in\b|$)",
    # "I own / I run / I operate a ..."
    r"\bI\s+(?:own|run|operate|manage)\s+(?:my\s+own\s+)?(?:a\s+|an\s+)?([\w][\w\s\-/]{2,35}?\s+(?:business|company|shop|salon|restaurant|clinic|store|agency|firm|practice))\b",
    # "I've been a ... for X years"
    r"\bI(?:'ve|\s+have)\s+been\s+(?:a\s+|an\s+)([\w][\w\s\-/]{2,40}?)\s+for\s+(?:\w+\s+)?(?:year|decade)",
    # "my work / my job / my career is (as) a ..."
    r"\bmy\s+(?:work|job|career|profession|field)\s+is\s+(?:in\s+|as\s+(?:a\s+|an\s+)?)?([\w][\w\s\-/]{2,40}?)(?:\.|,|\s+and\b|$)",
    # "I became a ... " — career endpoint phrasing
    r"\bI\s+became\s+(?:a\s+|an\s+)([\w][\w\s\-/]{2,40}?)(?:\s+(?:in\s+\d{4}|after|when|eventually|later)\b|\.|,|$)",
    # "I'm in [field]" — "I'm in medicine / engineering / education"
    r"\bI(?:'m|\s+am)\s+in\s+(medicine|engineering|education|technology|finance|law|social\s+work|healthcare|nursing|accounting|real\s+estate|construction|hospitality|the\s+military|the\s+military\s+service|the\s+restaurant\s+business|the\s+nail\s+(?:industry|business)|the\s+food\s+(?:industry|business))\b",
    # interviewer Q&A context: "what do you do?" → next turn answer
    r"\bwhat\s+(?:do\s+you\s+do|is\s+your\s+(?:current\s+)?(?:job|occupation|profession|work))\b[^.?\n]{0,150}?\bI\s+(?:am|work)\s+(?:a\s+|an\s+|as\s+(?:a\s+|an\s+))?([\w][\w\s\-/]{2,40}?)(?:\.|,|\n)",
]

# Simplified occupational tier mapping — keyword → tier
_FINAL_OCC_TIER = {
    # Professional / White-collar
    "doctor": "Professional", "physician": "Professional", "dentist": "Professional",
    "surgeon": "Professional", "psychiatrist": "Professional",
    "lawyer": "Professional", "attorney": "Professional",
    "engineer": "Professional", "software engineer": "Professional",
    "programmer": "Professional", "developer": "Professional",
    "pharmacist": "Professional", "optometrist": "Professional",
    "professor": "Professional", "instructor": "Professional",
    "teacher": "Professional", "educator": "Professional",
    "accountant": "Professional", "cpa": "Professional",
    "architect": "Professional",
    # Healthcare / Nursing
    "nurse": "Healthcare", "registered nurse": "Healthcare", "rn": "Healthcare",
    "social worker": "Healthcare/Social", "therapist": "Healthcare/Social",
    "counselor": "Healthcare/Social",
    # Business / Commerce
    "businessman": "Business", "businesswoman": "Business",
    "entrepreneur": "Business", "owner": "Business",
    "restaurant": "Business/Service", "nail salon": "Business/Service",
    "nail technician": "Business/Service", "nail": "Business/Service",
    "real estate": "Business", "realtor": "Business",
    # Technical / Skilled
    "technician": "Skilled/Technical", "mechanic": "Skilled/Technical",
    "electrician": "Skilled/Technical", "plumber": "Skilled/Technical",
    "contractor": "Skilled/Technical",
    # Government / Military
    "civil servant": "Government", "government worker": "Government",
    "military": "Military", "soldier": "Military", "officer": "Military",
    "police": "Government/Security", "firefighter": "Government/Security",
    # Service / Retail
    "cashier": "Service", "waiter": "Service", "waitress": "Service",
    "clerk": "Service", "customer service": "Service",
    # Retired
    "retired": "Retired",
    # Student (2nd-gen / young narrators)
    "student": "Student",
}


def _normalise_final_occ(raw: str | None) -> tuple[str | None, str | None]:
    """
    Returns (cleaned_title, tier) from a raw extracted string.
    Tier is looked up by longest matching keyword in _FINAL_OCC_TIER.
    """
    if not raw:
        return None, None
    cleaned = " ".join(raw.strip().split()).strip(".,;:")
    # Trim obvious over-captures (more than 5 words = probably sentence fragment)
    words = cleaned.split()
    if len(words) > 6:
        cleaned = " ".join(words[:6])
    cleaned_lower = cleaned.lower()
    # Match longest keyword first
    tier = None
    for kw in sorted(_FINAL_OCC_TIER, key=len, reverse=True):
        if kw in cleaned_lower:
            tier = _FINAL_OCC_TIER[kw]
            break
    return cleaned if cleaned else None, tier


_BAD_OCCUPATION_CAPTURES = {
    "before me", "too", "like a month after we got there", "that was my dream",
    "every sunday", "until today", "the whole time", "for the paper",
}


def _is_plausible_job_capture(text: str | None) -> bool:
    if not text:
        return False
    cleaned = " ".join(str(text).strip().split()).strip(".,;:").lower()
    if not cleaned or cleaned in _BAD_OCCUPATION_CAPTURES:
        return False
    if len(cleaned.split()) > 7:
        return False
    bad_starts = ("like ", "about ", "around ", "before ", "after ", "until ", "since ")
    if cleaned.startswith(bad_starts):
        return False
    return True


def extract_final_occupation_supp(narrator_text: str) -> tuple[str | None, str | None]:
    """
    Apply supplementary final-occupation patterns to narrator text.
    Returns (raw_title, tier).
    """
    for pat in _SUPP_FINAL_OCC_PATTERNS:
        m = re.search(pat, narrator_text, re.IGNORECASE | re.MULTILINE)
        if m:
            # last capture group has the occupation string
            raw = m.group(m.lastindex or 1)
            title, tier = _normalise_final_occ(raw)
            if title and _is_plausible_job_capture(title):
                return title, tier
    return None, None


# ---------------------------------------------------------------------------

## Block 6 · Education & Occupation Extraction

Extract highest education level and occupational tier from narrator dialogue.

In [42]:
# 1d. Education and Occupation patterns (used by supplementary_extraction_all)
# ---------------------------------------------------------------------------

# ── Highest Education ────────────────────────────────────────────────────────
_SUPP_EDUCATION_PATTERNS = [
    r"\bDoctor\s+[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ' -]{1,60}\b",
    r"\bI\s+got\s+to\s+(\d{1,2}(?:st|nd|rd|th)?\s+grade)\b",
    r"\b(?:I\s+was\s+in|I\s+finished)\s+(\d{1,2}(?:\s*(?:st|nd|rd|th))?\s+grade)\b",
    r"\b(?:grade\s+(one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve))\b",
    r"\bI\s+finished\s+(ninth|ninth\s+grade|ninth-grade)\b",
    r"\bI\s+was\s+in\s+the\s+(\d{1,2}(?:st|nd|rd|th)?\s+year\s+of\s+college)\b",
    r"\bmy\s+education\s+level\s+is\s+"
    r"(high\s+school|middle\s+school|elementary\s+school|college|university|some\s+college)\b",
    r"\b((?!(?:my|his|her)\s+education\s+level\b)[A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{1,35})\s+is\s+high\s+school\b",
    r"\b(?:received|got)\s+(?:my\s+|his\s+|her\s+)?(b\.?s\.?|b\.?a\.?|m\.?s\.?|m\.?a\.?)\b",
    r"\b(?:undergraduate\s*:\s*)?(b\.?a\.?|b\.?s\.?|m\.?a\.?|m\.?s\.?)\b",
    r"\b(?:did\s+my|finished\s+my|got\s+my)\s+(masters?|master(?:'s)?|bachelor(?:'s)?|ph\.?d\.?)\b",
    r"\b(medical\s+doctor)\b",
    r"\bI\s+study\s+(medicine|psychology|electronics?|electrical\s+engineering|pharmacy)\b",
    r"\bmy\s+background\s+is\s+in\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{2,40})\b",
    r"\b(?:earned|received|attained|got)\s+(?:an?\s+)?(electronic\s+technician(?:['’]s)?\s+certificate)\b",
    r"\b(pharmacy\s+technique)\b",
    r"\bwith\s+a\s+degree\s+in\s+(chemical\s+engineering|electrical\s+engineering|business\s+administration|legal\s+studies)\b",
    r"\b(?:went\s+to|entered)\s+(pharmacy\s+school|engineering\s+school|teachers[’']?\s+college|law\s+school)\b",
    r"\b(?:in\s+school,\s+)?(Cal\s+State\s+Fullerton|UNLV|University\s+of\s+Nevada,\s+Las\s+Vegas)\b",
    r"\b(?:boarding\s+school)\b",
    r"\b(?:night\s+school)\b",
    # Degree statements
    r"\b(?:I\s+)?(?:earned|received|got|completed|finished|graduated\s+with)\s+(?:a\s+|an\s+|my\s+)?"
    r"(bachelor(?:'s)?(?:\s+degree)?|master(?:'s)?(?:\s+degree)?|ph\.?d\.?|doctorate|associate(?:'s)?(?:\s+degree)?"
    r"|high\s+school\s+diploma|ged|mba|law\s+degree|medical\s+degree|nursing\s+degree)\b",
    # "I have a ... degree"
    r"\bI\s+have\s+(?:a\s+|an\s+|my\s+)?"
    r"(bachelor(?:'s)?|master(?:'s)?|ph\.?d\.?|doctorate|associate(?:'s)?|high\s+school\s+diploma"
    r"|mba|law\s+degree|medical\s+degree)\b",
    # "I graduated from ... university / college / high school"
    r"\b(?:I\s+)?graduated\s+from\s+(?:\w+\s+){0,4}?"
    r"(university|college|high\s+school|community\s+college|trade\s+school|vocational\s+school)\b",
    r"\b(?:I\s+)?graduat(?:ed|e)\s+at\s+the\s+(high\s+school|college|university)\b",
    r"\bI\s+graduated\s+high\s+school\b",
    r"\bafter\s+I\s+graduat(?:e|ed)\s+(?:my\s+)?high\s+school\b",
    r"\bjust\s+graduated\s+high\s+school\s+and\s+then\s+(?:came|come)\s+here\s+and\s+get\s+to\s+college\b",
    r"\bI\s+went\s+up\s+to\s+(high\s+school|college|university)\b",
    r"\bthe\s+last\s+time\s+I\s+went\s+to\s+school\s+was\s+in\s+(high\s+school|college|university)\b",
    r"\b(?:went|go)\s+to\s+school\b[^.?\n]{0,80}\b(up\s+to\s+high\s+school|high\s+school)\b",
    r"\b(?:I\s+)?(?:finished|finish|completed)\s+(?:\w+\s+){0,3}?(university|college|high\s+school)\b",
    r"\bstudy\s+(high\s+school)\b",
    r"\bhigh\s+school\s+up\s+to\s+\w+\s+years\b",
    r"\b(?:the\s+)?first\s+year\s+of\s+(college|university)\b",
    r"\b(?:passed|pass)\s+(?:my\s+)?college\s+entrance\s+exam\b",
    r"\b(?:I\s+)?(?:finished|finish|completed)\s+\d+\s+years?\s+(?:in|of)\s+(college|university|high\s+school)\b",
    r"\b(?:middle\s+school\s+at\s+9th\s+grade|9th\s+grade)\b",
    r"\b(?:got|received|earned)\s+(?:my\s+|his\s+|her\s+)?degree\s+at\s+(college|university)\b",
    r"\b(?:getting|get|got|obtaining|obtain(?:ed)?)\s+(?:his|her|my)\s+degree\s+at\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{2,60}?(?:college|university|cal\s+state))\b",
    r"\b(?:attended|attend(?:ed|ing)?)\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{2,60}?(?:high\s+school|college|university))\b",
    r"\b(elementary\s+school\s*,?\s*and\s+then\s+high\s+school|from\s+elementary\s+to\s+high\s+school)\b",
    r"\bI\s+just\s+finished\s+about\s+(middle\s+school|high\s+school)\b",
    r"\bI\s+attended\s+([A-ZÀ-Ỹ][A-ZÀ-Ỹa-zà-ỹ\s]{2,60}?high\s+school)\b",
    r"\b(?:went\s+to|attended)\s+(pharmacy\s+school|engineering\s+school|teachers[’']?\s+college|law\s+school)\b",
    r"\b(?:studied|study)\s+in\s+(national\s+technical\s+school|machine\s+shop|technical\s+school)\b",
    r"\b(?:graduated|graduate)\s+from\s+(national\s+technical\s+school|machine\s+shop|technical\s+school)\b",
    r"\bcontinued\s+with\s+my\s+study\b",
    # Interviewer Q&A pivot: "what is your highest level of education?"
    r"\b(?:highest\s+(?:level\s+of\s+)?education|highest\s+degree)[^.?\n]{0,120}?"
    r"(bachelor|master|ph\.?d|doctorate|associate|high\s+school|some\s+college|college\s+degree"
    r"|graduate\s+degree|undergraduate)\b",
    # "I went to / attended college / university"
    r"\b(?:I\s+)?(?:went\s+to|attended)\s+(?:\w+\s+)?(?:a\s+)?"
    r"(college|university|community\s+college|trade\s+school|vocational\s+school)\b",
    r"\b(?:obtaining|earned|received)\s+(?:his|her|my)\s+(m\.?d\.?|ph\.?d\.?|master(?:'s)?|bachelor(?:'s)?)\b",
    r"\b(didn['’]t\s+(?:get\s+to\s+go|go)\s+to\s+school|was\s+homeschooled|military\s+academy)\b",
    r"\b(go\s+to\s+school\s+in\s+the\s+morning|went\s+to\s+school\s+back\s+then)\b",
    r"\bhọc\s+(tiểu\s+học|trung\s+học)\b",
    r"\btrường\s+(tiểu\s+học|trung\s+học)\b",
]

_EDUCATION_LEVEL_MAP = {
    # Highest → lowest so first match wins
    "ph.d":               "Doctorate",
    "phd":                "Doctorate",
    "doctorate":          "Doctorate",
    "medical doctor":     "Doctorate/Professional",
    "medical degree":     "Doctorate/Professional",
    "law degree":         "Doctorate/Professional",
    "law school":         "Doctorate/Professional",
    "master":             "Master's",
    "mba":                "Master's",
    "bachelor":           "Bachelor's",
    "college":            "Bachelor's",
    "university":         "Bachelor's",
    "teachers' college":  "Bachelor's",
    "teachers college":   "Bachelor's",
    "associate":          "Associate's",
    "community college":  "Associate's",
    "trade school":       "Vocational/Trade",
    "vocational school":  "Vocational/Trade",
    "boarding school":    "Secondary",
    "night school":       "Secondary",
    "nursing degree":     "Nursing Diploma",
    "high school diploma":"High School",
    "high school":        "High School",
    "ged":                "High School (GED)",
    "some college":       "Some College",
    "middle school":      "Middle School",
    "elementary school":  "Primary",
    "pharmacy school":    "Bachelor's",
    "engineering school": "Bachelor's",
    "cal state fullerton":"Some College",
    "unlv":"Some College",
    "university of nevada, las vegas":"Some College",
    "tiểu học":           "Primary",
    "trung học":          "Secondary",
}

_EDUCATION_CANONICAL_MAP = {
    "ba": "BA",
    "b.a": "BA",
    "b.a.": "BA",
    "bachelor": "BA",
    "bachelors": "BA",
    "bachelor's": "BA",
    "bs": "BS",
    "b.s": "BS",
    "b.s.": "BS",
    "master": "MA/MS",
    "masters": "MA/MS",
    "master's": "MA/MS",
    "ma": "MA/MS",
    "m.a": "MA/MS",
    "m.a.": "MA/MS",
    "ma/ms": "MA/MS",
    "ms": "MA/MS",
    "m.s": "MA/MS",
    "m.s.": "MA/MS",
    "mba": "MA/MS",
    "phd": "PhD",
    "ph.d": "PhD",
    "ph.d.": "PhD",
    "doctorate": "Doctorate/Professional",
    "doctorate/professional": "Doctorate/Professional",
    "doctorate professional": "Doctorate/Professional",
    "doctorate / professional": "Doctorate/Professional",
    "associate": "Associate's",
    "associate's": "Associate's",
    "associates": "Associate's",
    "some college": "Some College",
    "high school": "High School",
    "high school ged": "High School (GED)",
    "high school (ged)": "High School (GED)",
    "middle school": "Middle School",
    "primary": "Primary",
    "secondary": "Secondary",
    "vocational/trade": "Vocational/Trade",
    "vocational trade": "Vocational/Trade",
    "no formal schooling": "No formal schooling",
    "n/a": "N/A",
}

_BAD_EDUCATION_CAPTURES = {
    "chemical engineering",
    "four",
    "go to school in the morning",
    "i think this",
    "management systems",
    "mei ying",
    "my education level",
    "national technical school",
}


def _normalise_education(raw: str | None) -> str | None:
    if not raw:
        return None
    raw_clean = " ".join(str(raw).strip().split())
    raw_lower = raw_clean.lower()
    raw_key = re.sub(r"[^a-z0-9+/()' ]+", "", raw_lower).strip()

    if raw_key in _BAD_EDUCATION_CAPTURES:
        return None
    if raw_key in _EDUCATION_CANONICAL_MAP:
        return _EDUCATION_CANONICAL_MAP[raw_key]

    word_grade_match = re.search(
        r"\bgrade\s+(one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)\b",
        raw_lower,
    )
    if word_grade_match:
        grade_map = {
            "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
            "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12,
        }
        grade = grade_map.get(word_grade_match.group(1))
        if grade is not None:
            if grade <= 5:
                return "Primary"
            if grade <= 8:
                return "Middle School"
            return "High School"
    grade_match = re.search(r"\b(\d{1,2})(?:st|nd|rd|th)?\s+grade\b", raw_lower)
    if grade_match:
        try:
            grade = int(grade_match.group(1))
        except Exception:
            grade = None
        if grade is not None:
            if grade <= 5:
                return "Primary"
            if grade <= 8:
                return "Middle School"
            if grade <= 12:
                return "High School"
    if re.search(r"\b\d{1,2}(?:st|nd|rd|th)?\s+year\s+of\s+college\b", raw_lower):
        return "Some College"
    if "get to college" in raw_lower or "went to college" in raw_lower:
        return "Some College"
    if "degree in chemical engineering" in raw_lower or "degree in electrical engineering" in raw_lower:
        return "Bachelor's"
    if "degree in business administration" in raw_lower or "legal studies" in raw_lower:
        return "Bachelor's"
    if raw_lower.startswith("doctor "):
        return "Doctorate/Professional"
    if "electronic technician" in raw_lower and "certificate" in raw_lower:
        return "Vocational/Trade"
    if "pharmacy technique" in raw_lower:
        return "Vocational/Trade"
    if "middle school at 9th grade" in raw_lower or raw_lower.strip() == "9th grade":
        return "High School"
    if "college entrance exam" in raw_lower:
        return "High School"
    if "continued with my study" in raw_lower:
        return "Some schooling"
    if "didn" in raw_lower and "school" in raw_lower:
        return "No formal schooling"
    if "homeschooled" in raw_lower:
        return "Homeschooled"
    if "military academy" in raw_lower:
        return "Military academy"
    for kw in sorted(_EDUCATION_LEVEL_MAP, key=len, reverse=True):
        if kw in raw_lower:
            return _EDUCATION_LEVEL_MAP[kw]
    return None


def standardise_highest_education(df: pd.DataFrame) -> pd.DataFrame:
    """Harmonize all Highest_Education values into a small canonical set."""
    df = df.copy()
    if "Highest_Education" not in df.columns:
        return df

    def _canon(v: object) -> object:
        if pd.isna(v):
            return v
        norm = _normalise_education(str(v))
        return norm if norm else pd.NA

    df["Highest_Education"] = df["Highest_Education"].map(_canon)
    return df


def apply_manual_highest_education_overrides(df: pd.DataFrame) -> pd.DataFrame:
    """Apply conservative interview-level education fixes from close reading."""
    df = df.copy()
    if "Interview_ID" not in df.columns or "Highest_Education" not in df.columns:
        return df

    overrides = {
        "VAOHP0015_F01": "BA",
        "VAOHP0113": "MA/MS",
        "VAOHP0116": "Doctorate/Professional",
        "VAOHP0170": "MA/MS",
        "VAOHP0249": "Vocational/Trade",
        "VAOHP0252": "BS",
        "VAOHP035": "PhD",
        "VAOHP0057_F01_Eng": "No formal schooling",
        "VAOHP0098": "Vocational/Trade",
        "VAOHP0056_F01_Eng": "N/A",
        "VAOHP0243_F01": "N/A",
    }
    for interview_id, value in overrides.items():
        mask = df["Interview_ID"].astype(str).str.strip() == interview_id
        df.loc[mask, "Highest_Education"] = value
    return df


def apply_manual_birthplace_overrides(df: pd.DataFrame) -> pd.DataFrame:
    """Apply conservative interview-level birthplace fixes from close reading."""
    df = df.copy()
    if "Interview_ID" not in df.columns or "Birthplace" not in df.columns:
        return df

    overrides = {
        "VAHF0002": "United States",
        "VAOHP0365": "Culver City, California",
        "VAHF0007": "N/A",
        "VAOHP0040.2": "N/A",
        "VAOHP0041": "N/A",
        "VAOHP0121_F01": "N/A",
        "VAOHP379": "N/A",
    }
    for interview_id, value in overrides.items():
        mask = df["Interview_ID"].astype(str).str.strip() == interview_id
        df.loc[mask, "Birthplace"] = value
    return df


def standardise_birthplace(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize Birthplace strings while preserving missing values."""
    df = df.copy()
    if "Birthplace" not in df.columns:
        return df

    def _canon(v: object) -> object:
        if pd.isna(v):
            return v
        norm = _normalise_birthplace(str(v))
        return norm if norm else pd.NA

    df["Birthplace"] = df["Birthplace"].map(_canon)
    return df


# ── First US Job ─────────────────────────────────────────────────────────────
_SUPP_FIRST_JOB_PATTERNS = [
    # "my first job in America / the US was ..."
    r"\bmy\s+first\s+(?:job|work|position)\s+(?:in\s+(?:America|the\s+US|the\s+States|this\s+country))?\s*"
    r"(?:was|is|as)\s+(?:a\s+|an\s+)?([\w][\w\s\-/]{2,35}?)(?:\.|,|\s+and\b|$)",
    # "first job was as / working as ..."
    r"\bfirst\s+job\s+(?:here\s+)?(?:was\s+(?:as\s+)?|working\s+as\s+)(?:a\s+|an\s+)?([\w][\w\s\-/]{2,35}?)(?:\.|,|\s+and\b|$)",
    # "I started working as / I got a job as ..."
    r"\b(?:I\s+)?(?:started?\s+working|began?\s+work(?:ing)?|got\s+(?:my\s+)?first\s+job)\s+"
    r"(?:as\s+)?(?:a\s+|an\s+)?([\w][\w\s\-/]{2,35}?)(?:\s+(?:in|at|for)\b|\.|,|$)",
    # "I worked at a factory / restaurant / nail salon ..."
    r"\bI\s+worked\s+(?:at|in)\s+(?:a\s+|an\s+|the\s+)?(factory|restaurant|nail\s+salon|donut\s+shop"
    r"|grocery\s+store|supermarket|hotel|laundry|dry\s+cleaner|fish\s+cannery|farm|warehouse)\b",
    # "when I first came here I was a / I worked as ..."
    r"\bwhen\s+(?:I\s+)?(?:first\s+)?(?:came\s+here|arrived|got\s+here)[^.]{0,80}"
    r"\bI\s+(?:was\s+(?:a\s+|an\s+)|worked\s+as\s+(?:a\s+|an\s+))?([\w][\w\s\-/]{2,35}?)(?:\.|,|$)",
    r"\b(?:daytime\s+I\s+go\s+to\s+school\s+and\s+)?nighttime\s+I\s+work\s+for\s+(?:a\s+|an\s+)?([\w][\w\s\-/]{2,35}?)(?:\.|,|$)",
]


# ---------------------------------------------------------------------------

## Block 7 · Derive Age at Arrival

Compute `Age_at_Arrival` from birth year and arrival year.

In [43]:
# 2. Derive Age_at_Arrival
# ---------------------------------------------------------------------------

def derive_age_at_arrival(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Preserve step1's richer extraction (departure-age fallback, direct text inference)
    # so we can fall back to it when arithmetic derivation fails.
    step1_age = df["Age_at_Arrival"].copy() if "Age_at_Arrival" in df.columns \
        else pd.Series([np.nan] * len(df), index=df.index)

    raw_age = np.where(
        df["Birth_Year"].notna() & df["Year_of_Arrival"].notna(),
        df["Year_of_Arrival"] - df["Birth_Year"],
        np.nan,
    )
    df["Age_at_Arrival_Raw"] = raw_age

    # Arithmetic result — null out implausible values (keep negatives in _Raw for
    # 2nd-generation detection in derive_generation).
    arithmetic = np.where(
        pd.notna(raw_age) & ((raw_age < 0) | (raw_age > 80)),
        np.nan,
        raw_age,
    )

    # Prefer arithmetic result; fall back to step1's value when arithmetic is NaN.
    df["Age_at_Arrival"] = np.where(pd.notna(arithmetic), arithmetic, step1_age)

    us_born = df.get("Birthplace", pd.Series(index=df.index, dtype=object)).map(_is_us_birthplace).fillna(False)
    df.loc[us_born & df["Age_at_Arrival"].isna(), "Age_at_Arrival"] = 0
    return df


# ---------------------------------------------------------------------------

## Block 8 · Derive Generational Status

Classify narrators as 1st, 1.5, or 2nd generation based on age at arrival (README §5.1).

In [44]:
# 3. Derive Generational_Status  (README §5.1)
# ---------------------------------------------------------------------------

def generational_status(age: float | None) -> str | None:
    if pd.isna(age):
        return None
    age = int(age)
    if age >= 18:
        return "1st generation"
    if 6 <= age <= 17:
        return "1.5 generation"
    if 0 <= age <= 5:
        return "1.75 generation"
    # negative → born in host country proxy
    return "2nd generation"


def derive_generation(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Preserve step1's value (may include life-stage-language inference).
    step1_gen = df["Generational_Status"].copy() if "Generational_Status" in df.columns \
        else pd.Series([None] * len(df), index=df.index)

    # Use Age_at_Arrival_Raw (preserves negatives) so 2nd generation (born in the
    # US, negative age-at-arrival) can be assigned even when the cleaned
    # Age_at_Arrival is NaN.
    age_for_gen = df["Age_at_Arrival_Raw"] if "Age_at_Arrival_Raw" in df.columns \
        else df["Age_at_Arrival"]
    derived = age_for_gen.apply(generational_status)

    us_born = df.get("Birthplace", pd.Series(index=df.index, dtype=object)).map(_is_us_birthplace).fillna(False)
    derived = pd.Series(derived, index=df.index)
    derived.loc[us_born] = "2nd generation"

    # Prefer freshly derived value; fall back to step1's richer inference.
    df["Generational_Status"] = derived.where(derived.notna(), step1_gen)
    return df


# ---------------------------------------------------------------------------

## Block 9 · Derive Arrival Wave

Assign narrators to Wave 1 (1975), Wave 2, Wave 3 (ODP-era), etc. based on arrival year (README §5.2).

In [45]:
# 4. Derive Arrival_Wave  (README §5.2)
# ---------------------------------------------------------------------------

def arrival_wave(year: float | None) -> str | None:
    if pd.isna(year):
        return None
    year = int(year)
    if year <= 1975:
        return "Wave 1 (≤1975)"
    if 1976 <= year <= 1985:
        return "Wave 2 (1976–1985)"
    if 1986 <= year <= 1995:
        return "Wave 3 (1986–1995)"
    return "Not eligible (>1995)"


# Map hand-coded raw labels → canonical labels (in case user typed
# a short form like "Wave 1 (1975)" or "Wave 3 (ODP-era)").
_RAW_WAVE_MAP = {
    "wave 1 (1975)"    : "Wave 1 (≤1975)",
    "wave 1"           : "Wave 1 (≤1975)",
    "wave 2 (1979)"    : "Wave 2 (1976–1985)",
    "wave 2"           : "Wave 2 (1976–1985)",
    "wave 3 (odp-era)" : "Wave 3 (1986–1995)",
    "wave 3 (odp)"     : "Wave 3 (1986–1995)",
    "wave 3"           : "Wave 3 (1986–1995)",
    "post-1995 (excluded)": "Not eligible (>1995)",
    "not eligible (>1995)": "Not eligible (>1995)",
}


def _harmonise_raw_wave(val: str | float) -> str | None:
    """Normalise a hand-coded wave label to the canonical form."""
    if pd.isna(val):
        return None
    key = str(val).strip().lower()
    return _RAW_WAVE_MAP.get(key, str(val).strip())   # keep as-is if unknown


def _infer_wave_from_cohort(row: pd.Series) -> str | None:
    """
    Conservative fallback for Arrival_Wave when Year_of_Arrival is still missing.

    Preferred logic:
    1. Use Refugee_Cohort as the first fallback layer.
    2. Only use Year_Left_Country to break ties or refine late-period cohorts.
    """
    cohort = row.get("Refugee_Cohort")
    if pd.notna(cohort):
        cohort_key = str(cohort).strip().lower()
        if cohort_key in {
            "wave 1 (fall of saigon)",
            "fall of saigon (1975)",
        }:
            return "Wave 1 (≤1975)"
        if cohort_key == "boat people":
            leave = row.get("Year_Left_Country")
            if pd.notna(leave):
                try:
                    leave = int(leave)
                except Exception:
                    leave = None
            else:
                leave = None
            if leave is not None and 1986 <= leave <= 1995:
                return "Wave 3 (1986–1995)"
            return "Wave 2 (1976–1985)"
        if cohort_key in {
            "odp",
            "ho (humanitarian operation)",
        }:
            return "Wave 3 (1986–1995)"
        if cohort_key == "camp resettlement":
            leave = row.get("Year_Left_Country")
            if pd.notna(leave):
                try:
                    leave = int(leave)
                except Exception:
                    leave = None
            else:
                leave = None
            if leave is not None:
                return arrival_wave(leave + 1) or "Wave 2 (1976–1985)"
            return "Wave 2 (1976–1985)"
    leave = row.get("Year_Left_Country")
    if pd.notna(leave):
        try:
            leave = int(leave)
        except Exception:
            leave = None
        if leave is not None:
            return arrival_wave(leave if leave == 1975 else leave + 1)
    return None


def _infer_wave_from_text(row: pd.Series) -> str | None:
    """
    Final conservative fallback for Arrival_Wave using direct route/program cues
    from any saved Step 1 text columns.
    """
    text = " ".join(
        str(row.get(col) or "")
        for col in [
            "life_map_text", "time_log_text", "field_notes_text", "header_text",
            "narrator_dialogue", "narrator_dialogue_pre_us_family_history",
            "narrator_dialogue_post_us_life",
        ]
    )
    if not text.strip():
        return None
    if re.search(r"\b(?:fall\s+of\s+saigon|april\s+30(?:th)?(?:,?\s*1975)?|uss\s+midway|camp\s+pendleton|guam|helicopter|airlift)\b", text, re.IGNORECASE):
        return "Wave 1 (≤1975)"
    if re.search(r"\b(?:boat\s+people|boat\s+person|small\s+boat|fishing\s+boat|pirates?|refugee\s+camp)\b", text, re.IGNORECASE):
        leave = row.get("Year_Left_Country")
        if pd.notna(leave):
            try:
                leave = int(leave)
            except Exception:
                leave = None
            if leave is not None and 1986 <= leave <= 1995:
                return "Wave 3 (1986–1995)"
        return "Wave 2 (1976–1985)"
    if re.search(r"\b(?:odp|orderly\s+departure\s+program|humanitarian\s+operation|re[- ]education\s+camp|political\s+prisoner|reunification\s+program|family\s+reunification)\b", text, re.IGNORECASE):
        return "Wave 3 (1986–1995)"
    return None


def derive_wave(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Derive Arrival_Wave from Year_of_Arrival (authoritative).
    2. For rows where Year_of_Arrival is missing but a wave was already
       coded manually in the raw file, fall back to the raw value
       (harmonised to the canonical label format).
    3. If wave is still missing, infer from Refugee_Cohort:
       - Fall of Saigon cohort -> Wave 1
       - Boat people -> Wave 2 by default
       - ODP / HO -> Wave 3
       - Year_Left_Country only refines edge cases.
    """
    df = df.copy()
    derived = df["Year_of_Arrival"].apply(arrival_wave)

    # Harmonise any pre-existing raw wave labels
    raw_waves = df["Arrival_Wave"].apply(_harmonise_raw_wave) \
        if "Arrival_Wave" in df.columns else pd.Series([None] * len(df))

    heuristic = df.apply(_infer_wave_from_cohort, axis=1)
    text_heuristic = df.apply(_infer_wave_from_text, axis=1)

    # Use derived where possible; then raw manual coding; then cohort fallback.
    df["Arrival_Wave"] = derived.where(derived.notna(), raw_waves)
    df["Arrival_Wave"] = df["Arrival_Wave"].where(df["Arrival_Wave"].notna(), heuristic)
    df["Arrival_Wave"] = df["Arrival_Wave"].where(df["Arrival_Wave"].notna(), text_heuristic)
    return df


def derive_migration_anchor(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a single summary timing anchor without collapsing the meaning of the
    original variables.

    - Prefer Year_of_Arrival when available.
    - Otherwise fall back to Year_Left_Country.
    """
    df = df.copy()

    anchor_year = []
    anchor_type = []

    for _, row in df.iterrows():
        arrival = row.get("Year_of_Arrival")
        leave = row.get("Year_Left_Country")

        if pd.notna(arrival):
            try:
                anchor_year.append(int(arrival))
                anchor_type.append("arrival")
                continue
            except Exception:
                pass

        if pd.notna(leave):
            try:
                anchor_year.append(int(leave))
                anchor_type.append("departure")
                continue
            except Exception:
                pass

        anchor_year.append(np.nan)
        anchor_type.append(None)

    df["Migration_Anchor_Year"] = anchor_year
    df["Migration_Anchor_Type"] = anchor_type
    return df


# ---------------------------------------------------------------------------

## Block 10 · QC Flags

Flag rows with suspicious or conflicting values for manual review (README §6.3).

In [46]:
# 5. QC flags  (README §6.3)
# ---------------------------------------------------------------------------

def apply_qc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    flags = []
    low_conf_birth_year_na_ids = {
        "VAOHP0121_F01",
        "VAOHP0066",
        "VAOHP0117",
        "VAOHP0040.2",
        "VAOHP0110_F01",
        "VAOHP0098",
    }
    low_conf_education_na_ids = {
        "VAHF0006",
        "VAHF0007",
        "VAHF0008_F01",
        "VAHF0012",
        "VAOHP0015_F01",
        "VAOHP0171",
        "VAOHP0377",
        "VAOHP0044",
        "VAOHP0052",
    }

    for _, row in df.iterrows():
        row_flags = []
        interview_id = str(row.get("Interview_ID", "")).strip()

        # Impossible age at arrival
        age = row.get("Age_at_Arrival")
        if pd.notna(age):
            if age < 0:
                row_flags.append("negative_age_at_arrival")
            if age > 80:
                row_flags.append("implausible_age_at_arrival_gt80")

        # Arrival year missing
        if pd.isna(row.get("Year_of_Arrival")):
            row_flags.append("missing_year_of_arrival")

        # Birth year missing
        if pd.isna(row.get("Birth_Year")):
            row_flags.append("missing_birth_year")
            if interview_id in low_conf_birth_year_na_ids:
                row_flags.append("birth_year_na_manual_review")

        # Highest education missing
        if pd.isna(row.get("Highest_Education")) and interview_id in low_conf_education_na_ids:
            row_flags.append("highest_education_na_manual_review")

        # Excluded by arrival cutoff
        if pd.notna(row.get("Year_of_Arrival")) and row["Year_of_Arrival"] > 1995:
            row_flags.append("excluded_post1995")

        # Implausible birth year
        by = row.get("Birth_Year")
        if pd.notna(by) and (by < 1880 or by > 2000):
            row_flags.append("implausible_birth_year")

        # Country of origin unknown
        if pd.isna(row.get("Country_of_Origin")) or row["Country_of_Origin"] == "Unknown":
            row_flags.append("missing_country_of_origin")

        flags.append("|".join(row_flags) if row_flags else "")

    df["QC_Flags"] = flags
    df["QC_Review"] = df["QC_Flags"].apply(lambda x: 1 if x else 0)
    return df


# ---------------------------------------------------------------------------

## Block 11 · Inclusion Filter

Keep only rows that meet minimum data-quality criteria for downstream analysis.

In [47]:
# 6. Inclusion filter
# ---------------------------------------------------------------------------

def apply_inclusion_filter(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns (included_df, excluded_df).
    Inclusion: Year_of_Arrival ≤ 1995  OR  Year_of_Arrival is missing
               (keep missings in for now; flag them).
    Hard exclusion: Year_of_Arrival > 1995.
    """
    excluded = df[df["Year_of_Arrival"] > 1995].copy() if df["Year_of_Arrival"].notna().any() else pd.DataFrame()
    included = df[~df.index.isin(excluded.index)].copy()
    return included, excluded


# ---------------------------------------------------------------------------

## Block 12 · Missingness Report

Count and report missing values per field across the full dataset.

In [48]:
# 7. Missingness report
# ---------------------------------------------------------------------------

def _has_qc_flag(value: object, flag: str) -> bool:
    if pd.isna(value):
        return False
    return flag in str(value).split("|")


def _is_resolved_na(row: pd.Series, field: str) -> bool:
    """Treat selected low-confidence fields as intentionally unresolved N/A in reports."""
    if field == "Birth_Year":
        return pd.isna(row.get("Birth_Year")) and _has_qc_flag(row.get("QC_Flags"), "birth_year_na_manual_review")
    if field == "Highest_Education":
        return pd.isna(row.get("Highest_Education")) and _has_qc_flag(row.get("QC_Flags"), "highest_education_na_manual_review")
    return False


def _field_missing_mask(df: pd.DataFrame, field: str) -> pd.Series:
    mask = df[field].isna().copy()
    if field in {"Birth_Year", "Highest_Education"} and "QC_Flags" in df.columns:
        resolved_na = df.apply(lambda row: _is_resolved_na(row, field), axis=1)
        mask = mask & ~resolved_na
    return mask


def _field_filled_count(df: pd.DataFrame, field: str) -> int:
    return int(len(df) - _field_missing_mask(df, field).sum())


def missingness_report(df: pd.DataFrame) -> None:
    key_vars = [
        # Core migration
        "Birth_Year", "Year_of_Arrival", "Age_at_Arrival",
        "Generational_Status", "Arrival_Wave",
        "Gender", "Country_of_Origin", "Birthplace",
        # Study outcome variables
        "Highest_Education",
    ]
    print("\n--- Missingness report ---")
    for var in key_vars:
        if var not in df.columns:
            continue
        n_miss = int(_field_missing_mask(df, var).sum())
        pct = n_miss / len(df) * 100
        print(f"  {var:<25}  missing: {n_miss:3d}/{len(df)}  ({pct:.0f}%)")


# ---------------------------------------------------------------------------

## Block 13 · Summary Table & Missing-Data Dashboard

Build a crosstab summary of key variables and print a visual missing-data dashboard.

In [49]:
# 8. Summary table
# ---------------------------------------------------------------------------

def print_summary(df: pd.DataFrame) -> None:
    print("\n--- Generational status distribution ---")
    print(df["Generational_Status"].value_counts(dropna=False).to_string())

    print("\n--- Arrival wave distribution ---")
    print(df["Arrival_Wave"].value_counts(dropna=False).to_string())

    print("\n--- Country of origin distribution ---")
    print(df["Country_of_Origin"].value_counts(dropna=False).to_string())

    print("\n--- Gender distribution ---")
    print(df["Gender"].value_counts(dropna=False).to_string())

    if df["Birth_Year"].notna().any():
        print(f"\n--- Birth year range: {int(df['Birth_Year'].min())} – {int(df['Birth_Year'].max())} ---")

    if df["Year_of_Arrival"].notna().any():
        print(f"--- Arrival year range: {int(df['Year_of_Arrival'].min())} – {int(df['Year_of_Arrival'].max())} ---")


def missingness_trace(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a per-narrator table showing which key fields are missing and
    what information IS available to guide manual entry.

    Prints a readable version and returns the DataFrame for inspection.
    """
    key_vars = [
        # Core migration
        "Birth_Year", "Year_of_Arrival", "Age_at_Arrival",
        "Generational_Status", "Arrival_Wave", "Gender", "Birthplace",
        # Study outcome variables (newly added)
        "Highest_Education",
    ]
    hint_vars = [
        "Narrator_Name", "Interview_Year", "Country_of_Origin",
        "Age_at_Arrival_Raw", "Year_Left_Country",
        "QC_Flags",
    ]

    rows = []
    for _, r in df.iterrows():
        missing = [v for v in key_vars if v in df.columns and pd.isna(r.get(v)) and not _is_resolved_na(r, v)]
        if not missing:
            continue
        hints = {h: r.get(h) for h in hint_vars if h in df.columns and pd.notna(r.get(h))}
        rows.append({
            "Interview_ID":   r.get("Interview_ID", r.get("Filename", "?")),
            "Narrator_Name":  r.get("Narrator_Name", ""),
            "Missing_Fields": ", ".join(missing),
            "N_Missing":      len(missing),
            **{h: v for h, v in hints.items() if h not in ("Narrator_Name",)},
        })

    trace_df = pd.DataFrame(rows).sort_values("N_Missing", ascending=False).reset_index(drop=True)

    print(f"\n--- Missingness trace: {len(trace_df)} narrators have at least one missing key field ---")
    print(f"{'ID':<30} {'Name':<25} {'Missing fields'}")
    print("-" * 100)
    for _, r in trace_df.iterrows():
        iid  = str(r["Interview_ID"])[:28]
        name = str(r.get("Narrator_Name", ""))[:23]
        print(f"  {iid:<30} {name:<25} {r['Missing_Fields']}")

    return trace_df


def missingness_verdict(df: pd.DataFrame, trace_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Classify each narrator with missing fields into a strategy group and
    print a plain-English verdict + action plan.

    Strategy groups
    ---------------
    A  both_missing_no_anchor   — Birth_Year + Year_of_Arrival missing, no Year_Left_Country
                                  → manual lookup required
    B  both_missing_has_anchor  — Birth_Year + Year_of_Arrival missing, Year_Left known
                                  → wave can be estimated from leave year; age still needs manual entry
    C  arrival_only_leave1975   — Year_of_Arrival missing, Year_Left = 1975
                                  → assign Wave 1 (≤1975) as fallback; Birth_Year present
    D  arrival_only_anchor      — Year_of_Arrival missing, Year_Left known (≠ 1975)
                                  → transit inference attempted; if still missing, estimate wave range
    E  birth_only               — Birth_Year missing, Year_of_Arrival available
                                  → search transcript for stated age-at-arrival; try back-calc
    F  birthplace_only          — only Birthplace missing
                                  → low priority; check transcript header manually
    G  gender_only              — only Gender missing
                                  → low priority; check honorifics / pronouns manually
    H  2nd_gen_confirmed        — Age_at_Arrival_Raw < 0, classified as 2nd generation
                                  → no action needed; code already handles
    """
    if trace_df.empty:
        print("\n--- Verdict: no missing key fields remaining ---")
        return df, trace_df

    verdict_rows = []

    for _, r in trace_df.iterrows():
        iid     = r.get("Interview_ID", "")
        missing = set(r["Missing_Fields"].split(", ")) if r["Missing_Fields"] else set()
        leave   = r.get("Year_Left_Country")
        raw_age = r.get("Age_at_Arrival_Raw")

        # --- Classify ---
        has_birth   = "Birth_Year"      not in missing
        has_arrival = "Year_of_Arrival" not in missing
        has_leave   = pd.notna(leave)

        # 2nd generation: born after arrival (negative raw age-at-arrival)
        if pd.notna(raw_age) and raw_age < 0 and missing <= {"Age_at_Arrival", "Generational_Status"}:
            strategy = "H_2nd_gen_confirmed"
            action   = "No action — 2nd generation, code assigns status from Age_at_Arrival_Raw."

        elif not has_birth and not has_arrival:
            if has_leave:
                strategy = "B_both_missing_has_anchor"
                leave_yr = int(leave)
                wave_est = ("Wave 1 (≤1975)" if leave_yr <= 1975
                            else "Wave 2 (1976–1985)" if leave_yr <= 1985
                            else "Wave 3 (1986–1995)")
                action   = (f"Year_Left={leave_yr} known → wave estimated as '{wave_est}'. "
                            "Still need Birth_Year + Year_of_Arrival from manual lookup "
                            "(archive record, intake form, or secondary source).")
            else:
                strategy = "A_both_missing_no_anchor"
                action   = ("No temporal anchor at all. Manual lookup required: "
                            "check collection finding aid, related family members' records, "
                            "or contact collection curator.")

        elif not has_arrival and has_birth:
            if has_leave and int(leave) == 1975:
                strategy = "C_arrival_only_leave1975"
                action   = ("Year_Left=1975 (Fall of Saigon). Assign Arrival_Wave='Wave 1 (≤1975)' "
                            "as fallback even without exact Year_of_Arrival. "
                            "Check transcript for transit camp duration to pin down year.")
            elif has_leave:
                strategy = "D_arrival_only_anchor"
                leave_yr = int(leave)
                wave_est = ("Wave 1 (≤1975)" if leave_yr <= 1975
                            else "Wave 2 (1976–1985)" if leave_yr <= 1985
                            else "Wave 3 (1986–1995)")
                action   = (f"Year_Left={leave_yr} → transit inference attempted. "
                            f"If still missing, wave range estimated as '{wave_est}'. "
                            "Search transcript for camp name + duration phrase.")
            else:
                strategy = "D_arrival_only_no_anchor"
                action   = ("No leave year either. Search transcript for duration phrases "
                            "('I've been here X years', 'came in the late 70s') "
                            "or decade references.")

        elif has_arrival and not has_birth:
            strategy = "E_birth_only"
            action   = ("Year_of_Arrival available. Search transcript for stated age-at-arrival "
                        "('I was X years old when I came') → back-calc Birth_Year = "
                        "Year_of_Arrival − age. Check collection record as secondary source.")

        elif missing == {"Birthplace"}:
            strategy = "F_birthplace_only"
            action   = ("Low priority. Check transcript header ('Place of Birth') and first "
                        "paragraph for city/province mention. Can be left missing if not found.")

        elif missing == {"Gender"}:
            strategy = "G_gender_only"
            action   = ("Check honorifics (Mr./Mrs./Ms.) and pronouns in transcript. "
                        "Vietnamese middle name 'Thị'=Female, 'Văn/Quốc/Hữu'=Male.")

        else:
            strategy = "X_mixed"
            action   = f"Mixed pattern ({r['Missing_Fields']}). Review transcript manually."

        verdict_rows.append({
            "Interview_ID":    iid,
            "Narrator_Name":   r.get("Narrator_Name", ""),
            "Missing_Fields":  r["Missing_Fields"],
            "N_Missing":       r["N_Missing"],
            "Year_Left":       leave,
            "Age_at_Arr_Raw":  raw_age,
            "Strategy_Group":  strategy,
            "Recommended_Action": action,
        })

    verdict_df = pd.DataFrame(verdict_rows).sort_values(
        ["Strategy_Group", "N_Missing"], ascending=[True, False]
    ).reset_index(drop=True)

    # --- Print verdict summary ---
    print("\n" + "=" * 80)
    print("MISSINGNESS VERDICT & STRATEGY")
    print("=" * 80)
    group_counts = verdict_df["Strategy_Group"].value_counts().sort_index()
    strategy_labels = {
        "A_both_missing_no_anchor":   "A — Both missing, no anchor   → manual lookup",
        "B_both_missing_has_anchor":  "B — Both missing, leave yr known → wave estimate + manual",
        "C_arrival_only_leave1975":   "C — Arrival missing, left 1975 → assign Wave 1 fallback",
        "D_arrival_only_anchor":      "D — Arrival missing, leave yr known → transit inference",
        "D_arrival_only_no_anchor":   "D — Arrival missing, no anchor → decade/duration search",
        "E_birth_only":               "E — Birth_Year missing only → back-calc from age-at-arrival",
        "F_birthplace_only":          "F — Birthplace only → low priority, header check",
        "G_gender_only":              "G — Gender only → honorifics/pronoun check",
        "H_2nd_gen_confirmed":        "H — 2nd generation (born in US) → no action needed",
        "X_mixed":                    "X — Mixed pattern → manual review",
    }
    for grp, count in group_counts.items():
        label = strategy_labels.get(grp, grp)
        print(f"  {count:3d} narrators  |  {label}")

    print("\n--- Action priority ---")
    print("  HIGH   : Groups A, B → manual lookup, no automation possible")
    print("  MEDIUM : Groups C, D → wave fallback assignable; exact year needs transcript check")
    print("  LOW    : Groups E, F, G → targeted transcript search or back-calculation")
    print("  DONE   : Group H → already handled by code")
    print("=" * 80)

    # --- Apply Wave 1 fallback for Group C (leave year = 1975, wave unresolved) ---
    group_c_ids = set(verdict_df.loc[
        verdict_df["Strategy_Group"] == "C_arrival_only_leave1975", "Interview_ID"
    ])
    if group_c_ids and "Arrival_Wave" in df.columns and "Interview_ID" in df.columns:
        mask = df["Interview_ID"].isin(group_c_ids) & df["Arrival_Wave"].isna()
        n_fixed = mask.sum()
        if n_fixed:
            df = df.copy()
            df.loc[mask, "Arrival_Wave"] = "Wave 1 (≤1975)"
            print(f"\nAuto-applied Wave 1 (≤1975) fallback to {n_fixed} Group-C narrators "
                  f"(Year_Left=1975, Arrival_Wave was missing).")

    # Return both the updated df (with any Wave-1 fallback applied) and the verdict
    return df, verdict_df


def variable_coverage_report(df: pd.DataFrame) -> None:
    """Coverage summary for the main study variables finalized in Step 2."""
    vars_to_check = [
        # Migration / demographics
        "Birth_Year", "Year_Left_Country", "Year_of_Arrival",
        "Migration_Anchor_Year",
        "Age_at_Arrival", "Generational_Status", "Arrival_Wave",
        "Gender", "Birthplace", "Country_of_Origin",
        # Education
        "Highest_Education",
        # Other
        "Refugee_Cohort",
    ]
    print("\n--- Step 2 variable coverage ---")
    for var in vars_to_check:
        if var not in df.columns:
            continue
        n = _field_filled_count(df, var)
        print(f"  {var:<28} {n:3d}/{len(df)}")


# ---------------------------------------------------------------------------
# Missing-data dashboard  (called at the end of main so you can see at a glance
# what still needs manual entry and which narrators are affected)
# ---------------------------------------------------------------------------

_DASHBOARD_FIELDS = [
    # Core migration
    "Birth_Year", "Year_of_Arrival", "Age_at_Arrival",
    "Generational_Status", "Arrival_Wave",
    "Gender", "Birthplace", "Interviewer",
    # Study outcome variables
    "Highest_Education",
]

def print_missing_dashboard(df: pd.DataFrame, verdict_df: pd.DataFrame) -> None:
    """
    Print a two-part dashboard:

    Part 1 — Field coverage bar chart
        Each key field gets a one-line filled/missing count with a visual bar.

    Part 2 — Per-narrator missing list grouped by strategy
        Shows exactly which Interview_ID / Narrator is missing which fields,
        grouped by the strategy group from the verdict so you know what action
        to take for each cluster.
    """
    total = len(df)
    W = 30   # bar width

    print("\n" + "=" * 70)
    print("  MISSING DATA DASHBOARD")
    print("=" * 70)

    # ── Part 1: field coverage bars ──────────────────────────────────────────
    print(f"\n  {'Field':<26}  {'Filled':>6}  {'Missing':>7}  Bar ({total} total)")
    print("  " + "-" * 66)
    for field in _DASHBOARD_FIELDS:
        if field not in df.columns:
            continue
        n_filled  = _field_filled_count(df, field)
        n_missing = total - n_filled
        filled_bars  = round(n_filled  / total * W)
        missing_bars = W - filled_bars
        bar = "█" * filled_bars + "░" * missing_bars
        flag = "  ← needs attention" if n_missing > 0 else ""
        print(f"  {field:<26}  {n_filled:>6}  {n_missing:>7}  [{bar}]{flag}")

    # ── Part 2: per-narrator list by strategy group ──────────────────────────
    if verdict_df is None or verdict_df.empty:
        print("\n  (No missingness verdict available — all fields complete.)")
        print("=" * 70)
        return

    print(f"\n  {'Narrators still needing manual entry':}")
    print("  " + "-" * 66)

    group_labels = {
        "A_both_missing_no_anchor":  "GROUP A — HIGH  | both Birth_Year + Arrival missing, no anchor → manual lookup",
        "B_both_missing_has_anchor": "GROUP B — HIGH  | both missing, leave year known → wave estimate + manual",
        "C_arrival_only_leave1975":  "GROUP C — MED   | arrival missing, left 1975 → assign Wave 1 fallback",
        "D_arrival_only_anchor":     "GROUP D — MED   | arrival missing, leave year known → transit search",
        "D_arrival_only_no_anchor":  "GROUP D — MED   | arrival missing, no anchor → decade/duration search",
        "E_birth_only":              "GROUP E — LOW   | birth year missing → back-calc from stated age",
        "F_birthplace_only":         "GROUP F — LOW   | birthplace only → check transcript header",
        "G_gender_only":             "GROUP G — LOW   | gender only → honorifics / pronouns",
        "H_2nd_gen_confirmed":       "GROUP H — DONE  | 2nd generation, code handles automatically",
        "X_mixed":                   "GROUP X — REVIEW| mixed missing pattern → manual review",
    }

    for grp in sorted(verdict_df["Strategy_Group"].unique()):
        sub = verdict_df[verdict_df["Strategy_Group"] == grp]
        label = group_labels.get(grp, f"GROUP {grp}")
        print(f"\n  {label}  ({len(sub)} narrators)")
        for _, r in sub.iterrows():
            iid  = str(r.get("Interview_ID", ""))[:28]
            name = str(r.get("Narrator_Name", ""))[:24]
            miss = str(r.get("Missing_Fields", ""))
            print(f"    {iid:<30} {name:<26} {miss}")

    print("\n" + "=" * 70)


# ---------------------------------------------------------------------------

## Block 14 · Column Order & Output Helpers

Define the final column ordering and save helpers for CSV / Excel output.

In [50]:
# Column order for final output
# ---------------------------------------------------------------------------

COLUMN_ORDER = [
    # Identification
    "Interview_ID", "Filename", "Narrator_Name", "Source", "Sub_Collection",
    "Interview_Year", "Interview_Date", "Interview_Location",
    "Interviewer", "Transcriber", "Translator", "Interview_Length",
    # Demographics
    "Country_of_Origin", "Gender", "Birth_Year", "Birthplace",
    # Migration
    "Year_Left_Country", "Year_of_Arrival", "Migration_Anchor_Year", "Migration_Anchor_Type",
    "Year_of_Arrival_Inferred", "Age_at_Arrival",
    "Generational_Status", "Arrival_Wave",
    # Education
    "Highest_Education",
    # QC
    "QC_Flags", "QC_Review",
    # Transcript stats
    "Num_Pages", "Num_Chars", "Filepath",
]

KEY_VARIABLE_COLUMN_ORDER = [
    "Interview_ID", "Narrator_Name",
    "Birth_Year", "Birthplace", "Gender",
    "Country_of_Origin",
    "Year_Left_Country", "Year_of_Arrival", "Migration_Anchor_Year", "Migration_Anchor_Type",
    "Age_at_Arrival", "Generational_Status", "Arrival_Wave", "Refugee_Cohort",
    "Highest_Education",
    "QC_Flags", "QC_Review",
]


# ---------------------------------------------------------------------------

## Block 15 · Run

Phase 1 — extract all variables · Phase 2 — fill missing · Phase 3 — report · Save outputs

In [51]:
# ── STEP 2: SCHEMA HARMONISER ─────────────────────────────────────────────
STEP1_SCHEMA_ALIASES = {
    # canonical text/context names used in Step 2
    "pre_america": "pre_us_family_history",
    "post_america": "post_us_life",
    "Field_Notes_Text": "field_notes_text",
    "Time_Log_Text": "time_log_text",
    "Life_Map_Text": "life_map_text",
    "header_text": "header_text",
    # canonical ID/meta names
    "interview_id": "Interview_ID",
    "filename": "Filename",
    "filepath": "Filepath",
    "source": "Source",
    "interview_year": "Interview_Year",
    "interview_date": "Interview_Date",
    "interview_location": "Interview_Location",
    "narrator_name": "Narrator_Name",
    "interviewer_name": "Interviewer",
    "sub-collection": "Sub_Collection",
    "interview_length": "Interview_Length",
    "num_chars": "Num_Chars",
    "num_pages": "Num_Pages",
}

STEP1_TEXT_COLUMNS = [
    "narrator_dialogue",
    "interviewer_dialogue",
    "pre_us_family_history",
    "post_us_life",
    "field_notes_text",
    "header_text",
    "time_log_text",
    "life_map_text",
]

STEP2_ANALYTIC_COLUMNS = [
    "Birth_Year",
    "Year_Left_Country",
    "Year_of_Arrival",
    "Year_of_Arrival_Inferred",
    "Interview_Year",
    "Age_at_Arrival",
    "Age_at_Arrival_Raw",
    "Generational_Status",
    "Arrival_Wave",
    "Birthplace",
    "Country_of_Origin",
    "Gender",
    "Refugee_Cohort",
    "Highest_Education",
    "Migration_Anchor_Year",
    "Migration_Anchor_Type",
    "QC_Flags",
    "QC_Review",
]

def harmonise_step1_schema(df):
    df = df.copy()

    # Prefer Step 1's canonical names if present; otherwise fall back to legacy names.
    for src, dest in STEP1_SCHEMA_ALIASES.items():
        if dest not in df.columns and src in df.columns:
            df[dest] = df[src]

    # Guarantee canonical transcript split columns from Step 1.
    if "pre_us_family_history" not in df.columns and "Pre_US_Family_History" in df.columns:
        df["pre_us_family_history"] = df["Pre_US_Family_History"]
    if "post_us_life" not in df.columns and "Post_US_Life" in df.columns:
        df["post_us_life"] = df["Post_US_Life"]

    # Aliases used inside the legacy Step 2 extraction code.
    if "narrator_dialogue_pre_us_family_history" not in df.columns and "pre_us_family_history" in df.columns:
        df["narrator_dialogue_pre_us_family_history"] = df["pre_us_family_history"]
    if "narrator_dialogue_post_us_life" not in df.columns and "post_us_life" in df.columns:
        df["narrator_dialogue_post_us_life"] = df["post_us_life"]

    # Backward compatibility for any remaining old references.
    if "narrator_dialogue_pre_america" not in df.columns and "pre_us_family_history" in df.columns:
        df["narrator_dialogue_pre_america"] = df["pre_us_family_history"]
    if "narrator_dialogue_post_america" not in df.columns and "post_us_life" in df.columns:
        df["narrator_dialogue_post_america"] = df["post_us_life"]

    return df


def ensure_step2_columns(df):
    df = harmonise_step1_schema(df)

    for col in STEP2_ANALYTIC_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    return df


def drop_duplicate_columns_keep_first(df, verbose=True):
    """Drop duplicate columns after normalising names by lowercase/strip.

    Keeps the first occurrence and drops later duplicates such as:
    Pre_US_Family_History vs pre_us_family_history
    Post_US_Life vs post_us_life
    """
    df = df.copy()

    original_cols = list(df.columns)
    normalised_cols = [str(c).strip().lower() for c in original_cols]

    seen = set()
    keep_mask = []
    dropped = []

    for original, normalised in zip(original_cols, normalised_cols):
        if normalised in seen:
            keep_mask.append(False)
            dropped.append(original)
        else:
            keep_mask.append(True)
            seen.add(normalised)

    if verbose and dropped:
        print("Dropping duplicate columns (keeping first occurrence):")
        for col in dropped:
            print(f"  - {col}")

    df = df.loc[:, keep_mask].copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def drop_redundant_alias_columns(df, verbose=True):
    """Drop legacy alias columns once canonical Step 2 columns exist.

    This removes columns like:
    - pre_america once pre_us_family_history exists
    - post_america once post_us_life exists
    - narrator_dialogue_pre_america once narrator_dialogue_pre_us_family_history exists
    - narrator_dialogue_post_america once narrator_dialogue_post_us_life exists
    - Pre_US_Family_History / Post_US_Life once lowercase canonical names exist
    """
    df = df.copy()

    alias_pairs = [
        ("pre_america", "pre_us_family_history"),
        ("post_america", "post_us_life"),
        ("Pre_US_Family_History", "pre_us_family_history"),
        ("Post_US_Life", "post_us_life"),
        ("narrator_dialogue_pre_america", "narrator_dialogue_pre_us_family_history"),
        ("narrator_dialogue_post_america", "narrator_dialogue_post_us_life"),
    ]

    to_drop = []
    for alias, canonical in alias_pairs:
        if alias in df.columns and canonical in df.columns:
            to_drop.append(alias)

    if verbose and to_drop:
        print("Dropping redundant alias columns:")
        for c in to_drop:
            print(f"  - {c}")

    return df.drop(columns=to_drop, errors="ignore")


In [52]:
# ── STEP 2: CLEAN TEXT COLUMNS ────────────────────────────────────────────
def prepare_clean_text_columns(df):
    df = df.copy()
    df = harmonise_step1_schema(df)

    text_cols = [
        "narrator_dialogue",
        "interviewer_dialogue",
        "pre_us_family_history",
        "post_us_life",
        "field_notes_text",
        "header_text",
        "time_log_text",
        "life_map_text",
        "narrator_dialogue_pre_us_family_history",
        "narrator_dialogue_post_us_life",
        "narrator_dialogue_pre_america",
        "narrator_dialogue_post_america",
    ]

    for col in text_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .fillna("")
                .astype(str)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
            )

    return df


In [53]:
print("Final INPUT_FILE used:", INPUT_FILE)
raw = load_step1_output(str(INPUT_FILE))
raw = ensure_step2_columns(raw)

print("\nCanonical text columns coverage:")
for c in ["narrator_dialogue", "pre_us_family_history", "post_us_life", "field_notes_text", "header_text", "time_log_text", "life_map_text"]:
    if c in raw.columns:
        n = int(raw[c].fillna("").astype(str).str.strip().ne("").sum())
        print(f"  {c:<28} {n}/{len(raw)}")
    else:
        print(f"  {c:<28} MISSING")


Final INPUT_FILE used: interview_data.xlsx
DEBUG load_step1_output path: interview_data.xlsx

Canonical text columns coverage:
  narrator_dialogue            230/230
  pre_us_family_history        MISSING
  post_us_life                 MISSING
  field_notes_text             MISSING
  header_text                  MISSING
  time_log_text                MISSING
  life_map_text                MISSING


In [54]:
# Single-pass supplementary extraction from the text already saved by Step 1
raw = supplementary_extraction_all(raw)
raw = standardise_birthplace(raw)
raw = apply_manual_birthplace_overrides(raw)
raw = standardise_highest_education(raw)
raw = apply_manual_highest_education_overrides(raw)

# Derived analytic variables
df = derive_age_at_arrival(raw)
df = derive_generation(df)
df = derive_wave(df)
df = derive_migration_anchor(df)

# QC + inclusion
df = apply_qc(df)
included, excluded = apply_inclusion_filter(df)

if len(excluded):
    print(f"\n{len(excluded)} record(s) excluded (Year_of_Arrival > 1995):")
    cols_to_show = [c for c in ["Filename", "Year_of_Arrival"] if c in excluded.columns]
    if cols_to_show:
        print(excluded[cols_to_show].to_string(index=False))

# Missingness trace + verdict
trace_df = missingness_trace(included)
included, verdict_df = missingness_verdict(included, trace_df)

# trace_df.to_csv(TRACE_FILE, index=False)
# print(f"\nSaved missingness trace   → {TRACE_FILE}")
# verdict_df.to_csv(VERDICT_FILE, index=False)
# print(f"Saved missingness verdict → {VERDICT_FILE}")

# Final schema cleanup
included = drop_redundant_alias_columns(included, verbose=True)
included = drop_duplicate_columns_keep_first(included, verbose=True)

# Reports
missingness_report(included)
variable_coverage_report(included)
print_summary(included)
print_missing_dashboard(included, verdict_df)

# Save outputs
extra_cols = [c for c in included.columns if c not in COLUMN_ORDER]
final_cols = [c for c in COLUMN_ORDER if c in included.columns] + extra_cols
included = included[final_cols]

included.to_excel(OUTPUT_FILE, index=False)
print(f"\nSaved {len(included)} records → {OUTPUT_FILE}")

# included.to_excel(ALL_VARIABLES_FILE, index=False)
# print(f"Saved all-variables file  → {ALL_VARIABLES_FILE}")

# key_cols = [c for c in KEY_VARIABLE_COLUMN_ORDER if c in included.columns]
# included[key_cols].to_excel(KEY_VARIABLES_FILE, index=False)
# print(f"Saved key-variables file  → {KEY_VARIABLES_FILE}")

# qc_rows = included[included["QC_Review"] == 1] if "QC_Review" in included.columns else pd.DataFrame()
# if len(qc_rows):
#     qc_rows.to_excel(FLAGS_FILE, index=False)
#     print(f"Saved {len(qc_rows)} QC-flagged rows → {FLAGS_FILE}")
# else:
#     print("No QC flags raised.")



  SUPPLEMENTARY EXTRACTION  (230 narrators have gaps)
  Birth_Year                   before:   0/230
  Year_Left_Country            before:   0/230
  Year_of_Arrival              before:   0/230
  Birthplace                   before:   0/230
  Country_of_Origin            before:   0/230
  Gender                       before:   0/230
  Refugee_Cohort               before:   0/230
  Interviewer                  before: 230/230
  Highest_Education            before:   0/230

  Recovered this pass:
    Birth_Year                   +135   total filled: 135/230
    Year_Left_Country            +40    total filled: 40/230
    Year_of_Arrival              +63    total filled: 63/230
    Birthplace                   +136   total filled: 136/230
    Country_of_Origin            +170   total filled: 170/230
    Gender                       +178   total filled: 178/230
    Refugee_Cohort               +174   total filled: 174/230
    Interviewer                  +0     total filled: 230/230
    